## 1. Environment, configuration, reproducibility

In [ ]:

import os
from pathlib import Path

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final")
KAGGLE_WORK_DIR = Path("/kaggle/working")

if not Path("/kaggle/input").exists():
    raise RuntimeError("This notebook version is for Kaggle (/kaggle/input not found). Use the Colab version elsewhere.")
print("Kaggle input folders:", sorted(p.name for p in Path("/kaggle/input").iterdir()))
print("Expected data dir exists:", KAGGLE_DATA_DIR.exists())


In [ ]:

import os
_d = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else None
if _d is not None:
    print(sorted(os.listdir(_d)))


In [ ]:
%pip install -q faster-whisper sentence-transformers sentencepiece soundfile

In [ ]:
import gc, hashlib, json, math, os, random, re, sys, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# ------------------------------------------------------------------ configuration
SEED = 42
N_SPLITS = 5
N_REPEATS = 3               # v4: repeated CV with different fold seeds (1 = v3 behaviour, faster)
CV_MODE = "speaker"         # "speaker": folds grouped by inferred speaker (v5) | "random": v1's folds (leaky) | "group": file_number // GROUP_SIZE
GROUP_SIZE = 3              # only used by CV_MODE="group"
SPEAKER_PERCENTILE = 99.5   # link two training clips when their voice similarity exceeds this percentile of the test->train similarities
VOICE_LAYERS = (3, 6)       # early SSL layers mostly encode who is speaking, not what is said
FORCE_RECOMPUTE = False     # True → ignore cached model predictions in data/interim/predictions

ASR_MODEL = "medium.en"     # faster-whisper model; changing it creates a new transcript cache
VERBATIM_PROMPT = False     # experimental: prompt Whisper to keep fillers/repetitions (needs re-transcription)
VERBATIM_PROMPT_TEXT = "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking."

LM_NAME = "gpt2-medium"     # language model used for perplexity features
SENTENCE_ENCODER = "sentence-transformers/all-mpnet-base-v2"

# Frozen audio backbones: every hidden layer is mean+std pooled over time, then probed layer-by-layer.
# "optional": True -> if the backbone fails to download/run, it is skipped with a message and the run continues.
AUDIO_BACKBONES = {
    "wavlm_base_plus":      {"kind": "ssl",     "name": "microsoft/wavlm-base-plus", "chunk_sec": 15},
    "wavlm_large":          {"kind": "ssl",     "name": "microsoft/wavlm-large",     "chunk_sec": 15},
    "whisper_medium_enc":   {"kind": "whisper", "name": "openai/whisper-medium.en",  "chunk_sec": 30},
    # --- v3 additions
    "whisper_large_v3_enc": {"kind": "whisper", "name": "openai/whisper-large-v3",   "chunk_sec": 30, "optional": True},
    "whisper_small_enc":    {"kind": "whisper", "name": "openai/whisper-small.en",   "chunk_sec": 30, "optional": True},
    # --- v4 additions
    "hubert_large":         {"kind": "ssl",     "name": "facebook/hubert-large-ll60k",            "chunk_sec": 15, "optional": True},
    "wav2vec2_asr_large":   {"kind": "ssl",     "name": "facebook/wav2vec2-large-960h-lv60-self", "chunk_sec": 15, "optional": True},
}

# Frozen text backbones (per-layer mean+std over tokens). dtype fp32 where fp16 can overflow / is unstable.
TEXT_BACKBONES = {
    "roberta_large":    {"name": "roberta-large",              "dtype": "fp16", "batch": 16},
    "qwen2_5_1_5b":     {"name": "Qwen/Qwen2.5-1.5B",          "dtype": "fp32", "batch": 8, "skip_first": True, "optional": True},
    "deberta_v3_large": {"name": "microsoft/deberta-v3-large", "dtype": "fp32", "batch": 8, "optional": True},
}
EXTRA_SENTENCE_ENCODERS = {"bge_large": "BAAI/bge-large-en-v1.5"}      # extra sentence-embedding SVR models (optional)
USE_ELECTRA = True                                                       # ELECTRA replaced-token features (optional)
ELECTRA_NAME = "google/electra-large-discriminator"

# Probe heads fitted on every backbone (one shared layer ranking per fold): svr | ridge | krr | lstack
PROBE_HEADS = ("svr", "ridge", "krr", "lstack")
PROBE_VERSION = "v5.0"      # part of every prediction-cache key; bump it if you edit the probe/stack code

RUN_DEBERTA = False         # reference: DeBERTa-v3 fine-tuning gave 0.69 RMSE under grouped CV and was dropped; set True to run it (~20 min)
DEBERTA = dict(name="microsoft/deberta-v3-base", max_len=320, epochs=5, lr=2e-5, batch=8, accum=2)

V1_REFERENCE = {"rmse": 0.532446, "pearson": 0.851207}   # v1 final nested-OOF numbers, for comparison only
V2_REFERENCE = {"rmse": 0.4956, "pearson": 0.8726}        # v2 final nested-OOF numbers (RANDOM folds - not comparable with speaker-grouped CV)
REFERENCE_NB = {"grouped_cv_rmse": 0.5094, "pearson": 0.8646, "public_lb": 0.3382}   # reference notebook: speaker-grouped nested-stack CV on graded clips, public LB

# ---- v5 additions (ideas from the reference notebook)
USE_ZERO_GATE = True        # set test clips that are "loud from start to finish" (like the 37 grade-0 training clips) to 0
USE_BATCH_STACK = True      # also try stackers with separate (partially pooled) weights for the ~45 s recording batch
SELECT_BY = "testmix"       # choose the stacker by "testmix" (nested RMSE re-weighted to the TEST batch mix) or "plain" nested RMSE
# Whisper DECODER hidden states over the ASR transcript, conditioned on the audio (optional; probed like the audio backbones)
DECODER_BACKBONES = {
    "whisper_medium_dec": {"name": "openai/whisper-medium.en", "multilingual": False, "chunk_sec": 30, "optional": True},
    # "whisper_large_v3_dec": {"name": "openai/whisper-large-v3", "multilingual": True, "chunk_sec": 30, "optional": True},   # slower, ~2x
}

IN_KAGGLE = os.path.exists("/kaggle/input")


def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


seed_everything()

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Kaggle: Settings → Accelerator → GPU T4 x2 (or P100), then run again.")
device = torch.device("cuda")
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)} | Kaggle: {IN_KAGGLE}")

In [ ]:
def locate_dataset():
    bases = [KAGGLE_DATA_DIR]
    if os.environ.get("GRAMMAR_DATA_DIR"):
        bases.insert(0, Path(os.environ["GRAMMAR_DATA_DIR"]))
    bases += [p.parent for p in Path("/kaggle/input").glob("**/train.csv")]
    for base in bases:
        if all((base / n).exists() for n in ("train.csv", "test.csv", "train", "test")):
            return base.resolve()
    raise FileNotFoundError(
        "Could not find the dataset. Attach the competition data to this notebook (Add Input) or set "
        "os.environ['GRAMMAR_DATA_DIR'] to the folder containing train.csv, test.csv, train/ and test/."
    )


DATA_DIR = locate_dataset()
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"

# /kaggle/input is read-only -> every cache and output goes to /kaggle/working
PROJECT_DIR = KAGGLE_WORK_DIR
CACHE_DIR, OUT_DIR = PROJECT_DIR / "cache", PROJECT_DIR / "outputs"
for d in (CACHE_DIR, OUT_DIR, CACHE_DIR / "predictions"):
    d.mkdir(parents=True, exist_ok=True)

# Optional warm start: copy caches from any attached Kaggle Dataset (never overwrites existing files).
import shutil
_seeded = 0
for _f in list(Path("/kaggle/input").glob("**/asr_transcripts_*.csv")) + \
          list(Path("/kaggle/input").glob("**/cache/**/*.npy")) + list(Path("/kaggle/input").glob("**/cache/**/*.csv")):
    try:
        _rel = _f.name if _f.name.startswith("asr_transcripts_") else _f.relative_to(next(p for p in _f.parents if p.name == "cache"))
        _dst = CACHE_DIR / _rel
        if not _dst.exists():
            _dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(_f, _dst); _seeded += 1
    except Exception as _err:  # noqa: BLE001
        print("cache seed skipped:", _f, repr(_err))
if _seeded:
    print(f"Seeded {_seeded} cache file(s) from attached Kaggle inputs")

print("Dataset :", DATA_DIR)
print("Caches  :", CACHE_DIR)
print("Outputs :", OUT_DIR)


## 2. Data audit

Two dataset quirks found in v1 are kept, because they materially change the modelling:
1. **Labels equal to `0`** fall outside the 1–5 rubric, come from a separate filename range and have much lower ASR confidence. The grammar regressor is trained on the rubric-valid rows only (`label > 0`). Section 12 re-checks whether any *test* clip looks like that regime.
2. **`sample_submission.csv` may not match `test.csv`** (204 vs 216 rows in v1's run). `test.csv` is checked against the audio files and used as the submission order.

Train and test reuse file names for different audio, so every cache is keyed by position / `(split, filename)`, never by file name alone.

In [ ]:
import soundfile as sf

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert {"filename", "label"} <= set(train_df.columns), train_df.columns.tolist()
assert "filename" in test_df.columns
assert train_df["filename"].is_unique and test_df["filename"].is_unique
assert train_df["label"].notna().all()
missing = [f for f in train_df["filename"] if not (TRAIN_DIR / f).exists()] + \
          [f for f in test_df["filename"] if not (TEST_DIR / f).exists()]
assert not missing, f"{len(missing)} audio files listed in the CSVs are missing, e.g. {missing[:3]}"

N_TRAIN, N_TEST = len(train_df), len(test_df)
nz_mask = train_df["label"].values > 0          # rubric-valid rows used for modelling
idx_nz = np.where(nz_mask)[0]

info = []
for split, frame, folder in (("train", train_df, TRAIN_DIR), ("test", test_df, TEST_DIR)):
    for f in frame["filename"]:
        i = sf.info(folder / f)
        info.append({"split": split, "filename": f, "duration": i.duration, "sr": i.samplerate, "channels": i.channels})
audio_info = pd.DataFrame(info)

print(f"train={N_TRAIN}  test={N_TEST}  sample_submission={len(sample_df)}  zero-label rows={int((~nz_mask).sum())}")
print(f"duration {audio_info.duration.min():.1f}–{audio_info.duration.max():.1f}s | sample rates {sorted(audio_info.sr.unique())}")
if set(sample_df.iloc[:, 0]) != set(test_df["filename"]):
    print("WARNING: sample_submission.csv differs from test.csv — submission is built in test.csv order (see Section 10).")

counts = train_df["label"].value_counts().sort_index()
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(counts.index.astype(str), counts.values, color="#4472C4")
ax[0].set(title="Training label distribution", xlabel="Grammar score", ylabel="Clips")
for s, c in (("train", "#4472C4"), ("test", "#ED7D31")):
    ax[1].hist(audio_info.loc[audio_info.split == s, "duration"], bins=20, alpha=0.7, label=s, color=c)
ax[1].set(title="Clip duration", xlabel="Seconds", ylabel="Clips"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3. Whisper transcription (resumable)

Grammar shows up in words and syntax, so each clip is transcribed with `faster-whisper`. Besides the text, the cache stores word-level timestamps and confidences, which Section 4 turns into pause and fluency features.
The cache is written every 20 clips and has the **same schema as v1**, so a v1 transcript file can be reused as-is.

In [ ]:
ASR_PATH = CACHE_DIR / f"asr_transcripts_{ASR_MODEL.replace('.', '_')}.csv"
EMPTY_ROW = {"transcript": "", "detected_language": "", "language_probability": np.nan, "audio_duration": np.nan,
             "segment_count": 0, "word_count": 0, "char_count": 0, "avg_logprob": np.nan, "avg_no_speech_prob": np.nan,
             "mean_word_probability": np.nan, "first_word_start": np.nan, "last_word_end": np.nan,
             "speech_span": 0.0, "words_json": "[]"}


from math import gcd
from scipy.signal import resample_poly

TARGET_SR = 16_000


def load_audio_16k(path):
    """Mono float32 audio at 16 kHz (resamples if a file is not already 16 kHz). Used by ASR and the encoders."""
    audio, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr != TARGET_SR:
        g = gcd(int(sr), TARGET_SR)
        audio = resample_poly(audio, TARGET_SR // g, int(sr) // g).astype(np.float32)
    return audio



def load_whisper():
    """float16 first; fall back to int8_float16 if the GPU/cuDNN build rejects it."""
    from faster_whisper import WhisperModel
    last_err = None
    for compute_type in ("float16", "int8_float16"):
        try:
            return WhisperModel(ASR_MODEL, device="cuda", compute_type=compute_type,
                                download_root=str(CACHE_DIR / "whisper_models"))
        except Exception as err:  # noqa: BLE001
            last_err = err
            print(f"Whisper load with {compute_type} failed: {err!r}")
    raise last_err


def transcribe_file(model, path):
    audio = load_audio_16k(path)      # decode with soundfile, so faster-whisper's PyAV decoder is never used
    segments, info = model.transcribe(
        audio, language="en", beam_size=5, vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 500}, word_timestamps=True,
        initial_prompt=VERBATIM_PROMPT_TEXT if VERBATIM_PROMPT else None)
    segments = list(segments)
    words = [{"word": w.word.strip(), "start": float(w.start), "end": float(w.end), "probability": float(w.probability)}
             for s in segments for w in (s.words or [])]
    text = " ".join(s.text.strip() for s in segments if s.text.strip())
    return {
        "transcript": text, "detected_language": info.language, "language_probability": float(info.language_probability),
        "audio_duration": float(info.duration), "segment_count": len(segments), "word_count": len(words),
        "char_count": len(text),
        "avg_logprob": float(np.mean([s.avg_logprob for s in segments])) if segments else np.nan,
        "avg_no_speech_prob": float(np.mean([s.no_speech_prob for s in segments])) if segments else np.nan,
        "mean_word_probability": float(np.mean([w["probability"] for w in words])) if words else np.nan,
        "first_word_start": words[0]["start"] if words else np.nan,
        "last_word_end": words[-1]["end"] if words else np.nan,
        "speech_span": (words[-1]["end"] - words[0]["start"]) if words else 0.0,
        "words_json": json.dumps(words, ensure_ascii=False),
    }


jobs = [("train", f, TRAIN_DIR / f) for f in train_df["filename"]] + [("test", f, TEST_DIR / f) for f in test_df["filename"]]
records = pd.read_csv(ASR_PATH).to_dict("records") if ASR_PATH.exists() else []
records = [r for r in records if not (isinstance(r.get("error"), str) and r["error"])]   # failed clips are retried
done = {(r["split"], r["filename"]) for r in records}
todo = [j for j in jobs if (j[0], j[1]) not in done]
print(f"ASR cache: {len(done)} done, {len(todo)} to transcribe ({ASR_PATH.name})")

if todo:
    whisper = load_whisper()
    n_errors = 0
    for n, (split, fname, path) in enumerate(tqdm(todo, desc="Whisper"), start=1):
        row = {"split": split, "filename": fname, "error": ""}
        try:
            row.update(transcribe_file(whisper, path))
        except Exception as err:  # noqa: BLE001  keep going; the clip gets an empty transcript
            n_errors += 1
            print("ASR error:", split, fname, repr(err))
            if n_errors >= 5 and n_errors == n:
                raise RuntimeError("The first 5 clips all failed, so something is wrong with the setup (see the error above). "
                                   "Nothing was saved to the cache.") from err
            row.update(EMPTY_ROW); row["error"] = repr(err)
        records.append(row)
        if n % 20 == 0 or n == len(todo):
            pd.DataFrame(records).to_csv(ASR_PATH, index=False)
    del whisper
    free_gpu()

In [ ]:
asr = pd.read_csv(ASR_PATH)
asr["transcript"] = asr["transcript"].fillna("").astype(str)
train_asr = train_df.merge(asr[asr.split == "train"].drop(columns="split"), on="filename", how="left", validate="one_to_one")
test_asr = test_df.drop(columns=[c for c in test_df.columns if c not in ("filename",)]).merge(
    asr[asr.split == "test"].drop(columns="split"), on="filename", how="left", validate="one_to_one")
assert len(train_asr) == N_TRAIN and len(test_asr) == N_TEST
assert train_asr["word_count"].notna().all() and test_asr["word_count"].notna().all(), "ASR cache incomplete"
train_asr["transcript"] = train_asr["transcript"].fillna("")
test_asr["transcript"] = test_asr["transcript"].fillna("")

# Every transcript-derived cache is tagged with a hash of the transcripts, so changing ASR settings invalidates them.
TAG = hashlib.md5("\n".join(train_asr["transcript"].tolist() + test_asr["transcript"].tolist()).encode()).hexdigest()[:8]
print("Transcript tag:", TAG)
print("Empty transcripts  train/test:", int((train_asr.transcript.str.strip() == "").sum()),
      "/", int((test_asr.transcript.str.strip() == "").sum()))

print("\nASR summary by label (note how the score-0 regime stands out):")
print(train_asr.groupby("label").agg(n=("label", "size"), words=("word_count", "mean"),
                                     word_prob=("mean_word_probability", "mean"),
                                     avg_logprob=("avg_logprob", "mean")).round(3).to_string())
rng = np.random.RandomState(SEED)
for score in (2.0, 3.0, 4.0, 5.0):
    pool = train_asr[train_asr.label == score]
    if len(pool):
        ex = pool.iloc[rng.randint(len(pool))]
        print(f"\n[score {score:g}] {ex.filename}: {ex.transcript[:350]}")

## 3b. Recording batches and the grade-0 gate  *(new in v5 — from the reference notebook)*

Two facts about the data that v4 did not use:

1. **Recording batches.** Clip duration reveals a separate batch of clips that are exactly ≈45.06 s long. It is a small share of the training set but a large share of the test set, and its grades are lower on average — the test set is **not** distributed like the training set. v5 therefore reports error per batch, selects the stacker on a *test-mix-weighted* nested RMSE, and can fit separate (partially pooled) stack weights for this batch.
2. **Grade-0 clips are recorded differently.** The clips graded 0 are loud from start to finish. The *speech ratio* — the share of 25 ms frames whose RMS exceeds 0.02 — is ≈1 only for them. The models are still trained on graded clips only; any **test** clip above the gate threshold is set to 0 in the submission. The threshold is derived from the training data, and the gate switches itself **off** unless it separates the training zeros from every graded clip perfectly.

Check the printed list of gated test clips. A clip wrongly set to 0 costs roughly (its grade)² / 216 of the test MSE, so look at it (and its neighbours in the speech-ratio ranking) before trusting the gate.

In [ ]:
def batch_of(d):
    """0 = the ~45.06 s recording batch | 1 = ~60 s | 2 = shorter than 44 s | 3 = 44-59 s."""
    return np.where(np.abs(d - 45.06) < 0.1, 0, np.where(d > 59.5, 1, np.where(d < 44, 2, 3)))


assert audio_info["filename"].tolist() == train_df["filename"].tolist() + test_df["filename"].tolist(), "audio_info order changed"
DURATION = audio_info["duration"].to_numpy(float)                 # train rows first, then test rows
BATCH = batch_of(DURATION)
BATCH_NAMES = ["45 s", "60 s", "< 44 s", "44-59 s"]
lab_all = train_df["label"].to_numpy(float)
TEST_SHARE = np.bincount(BATCH[N_TRAIN:], minlength=4) / N_TEST
batch_table = pd.DataFrame({
    "train share": np.bincount(BATCH[:N_TRAIN], minlength=4) / N_TRAIN,
    "test share": TEST_SHARE,
    "mean grade (train, >0)": [lab_all[(BATCH[:N_TRAIN] == b) & (lab_all > 0)].mean() if ((BATCH[:N_TRAIN] == b) & (lab_all > 0)).any() else np.nan
                               for b in range(4)]}, index=BATCH_NAMES)
print(batch_table.round(3).to_string())
if TEST_SHARE[0] > 0.25 and batch_table.iloc[0, 0] < 0.25:
    print("-> the 45 s batch is much more common in test than in train: per-batch errors and test-mix weighting matter.")

# ---- speech ratio: share of 25 ms frames with RMS > 0.02 (cached) ----
SR_PATH = CACHE_DIR / "speech_ratio.npy"


def speech_ratio_of(path, frame=400, thr=0.02):
    a = load_audio_16k(path)
    n = len(a) // frame
    if n == 0:
        return 0.0
    rms = np.sqrt((a[:n * frame].reshape(n, frame) ** 2).mean(1))
    return float((rms > thr).mean())


if SR_PATH.exists():
    SPEECH_RATIO = np.load(SR_PATH)
else:
    _paths = [TRAIN_DIR / f for f in train_df["filename"]] + [TEST_DIR / f for f in test_df["filename"]]
    SPEECH_RATIO = np.array([speech_ratio_of(p) for p in tqdm(_paths, desc="speech ratio")])
    np.save(SR_PATH, SPEECH_RATIO)
assert len(SPEECH_RATIO) == N_TRAIN + N_TEST

sr_tr, sr_te = SPEECH_RATIO[:N_TRAIN], SPEECH_RATIO[N_TRAIN:]
is_zero_tr = lab_all == 0
GATE, GATE_THR = np.zeros(N_TRAIN + N_TEST, dtype=bool), None
if USE_ZERO_GATE and is_zero_tr.any() and (~is_zero_tr).any():
    lo_zero, hi_graded = float(sr_tr[is_zero_tr].min()), float(sr_tr[~is_zero_tr].max())
    print(f"speech ratio: grade-0 training clips >= {lo_zero:.3f} | graded training clips <= {hi_graded:.3f}")
    if lo_zero > hi_graded:
        GATE_THR = max(0.5 * (lo_zero + hi_graded), lo_zero - 0.02)      # perfectly separable: stay close to the zero group
        GATE = SPEECH_RATIO >= GATE_THR
        print(f"gate threshold {GATE_THR:.3f}: training zeros caught {int((GATE[:N_TRAIN] & is_zero_tr).sum())}/{int(is_zero_tr.sum())}, "
              f"false positives {int((GATE[:N_TRAIN] & ~is_zero_tr).sum())}")
        gated = test_df["filename"][GATE[N_TRAIN:]].tolist()
        top_te = np.sort(sr_te)[::-1][:5]
        print(f"gated TEST clips ({len(gated)}): {gated}  |  highest test speech ratios: {np.round(top_te, 3).tolist()}")
        if len(gated) > 0.05 * N_TEST:
            print("WARNING: more than 5% of the test clips are gated - inspect them before trusting the gate.")
    else:
        print("WARNING: the speech ratio does not separate the grade-0 clips from graded clips here - gate switched OFF.")
else:
    print("zero gate disabled (USE_ZERO_GATE=False or no zero-label rows).")

frac_lt2 = float(((lab_all > 0) & (lab_all < 2)).sum() / max((lab_all > 0).sum(), 1))
CLIP_LO = 2.0 if frac_lt2 < 0.015 else 1.0                     # reference: grades below 2 are <1% of graded clips -> never predict below 2
print(f"graded training clips with grade < 2: {frac_lt2:.2%} -> predictions are clipped to [{CLIP_LO:g}, 5]")

plt.figure(figsize=(6, 3))
plt.scatter(sr_tr, lab_all, s=8, alpha=0.5)
if GATE_THR is not None:
    plt.axvline(GATE_THR, color="r", ls="--", label="gate")
    plt.legend(fontsize=8)
plt.xlabel("speech ratio"); plt.ylabel("grade (train)")
plt.tight_layout(); plt.show()

## 4. Interpretable transcript features

Four families, all computed from the transcript or the word timestamps:
* **Basic & lexical** (v1 set): length, ASR confidence, type/token ratio, fillers, conjunctions, sentence length.
* **Fluency** (new): pause counts/lengths, articulation rate, low-confidence word ratio — hesitation and repairs often co-occur with grammatical trouble.
* **Language-model surprise** (new): token/sentence negative log-likelihood under GPT-2. Ungrammatical text is more "surprising".
* **Syntax** (new, spaCy): sentences without a subject/finite verb (rubric level 2 "incomplete sentences"), parse depth, subordinate clauses, passives, POS mix (rubric level 5 "complex grammar").

Missing values are imputed with **training-set** medians only (v1 imputed test with test medians).

In [ ]:
WORD_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?")
FILLERS = {"um", "uh", "erm", "hmm", "ah", "huh"}
SUBORDINATORS = {"because", "although", "though", "while", "whereas", "unless", "since", "until", "before", "after",
                 "when", "whenever", "where", "wherever", "if", "whether"}
COORDINATORS = {"and", "but", "or", "so", "yet"}
ASR_RAW_COLS = ["word_count", "char_count", "segment_count", "audio_duration", "avg_logprob", "avg_no_speech_prob",
                "mean_word_probability", "first_word_start", "last_word_end", "speech_span"]


def linguistic_features(text):
    words = [w.lower() for w in WORD_RE.findall(str(text))]
    n = len(words)
    keys = ["lex_unique_ratio", "avg_word_length", "long_word_ratio", "filler_ratio", "subordinator_ratio",
            "coordinator_ratio", "repeat_ratio", "sentence_count", "words_per_sentence", "question_count"]
    if n == 0:
        return {k: 0.0 for k in keys}
    sentences = [s for s in re.split(r"[.!?]+", str(text)) if s.strip()]
    return {
        "lex_unique_ratio": len(set(words)) / n,
        "avg_word_length": float(np.mean([len(w) for w in words])),
        "long_word_ratio": float(np.mean([len(w) >= 7 for w in words])),
        "filler_ratio": sum(w in FILLERS for w in words) / n,
        "subordinator_ratio": sum(w in SUBORDINATORS for w in words) / n,
        "coordinator_ratio": sum(w in COORDINATORS for w in words) / n,
        "repeat_ratio": sum(words[i] == words[i - 1] for i in range(1, n)) / max(n - 1, 1),
        "sentence_count": float(len(sentences)),
        "words_per_sentence": n / max(len(sentences), 1),
        "question_count": float(str(text).count("?")),
    }


FLUENCY_KEYS = ["pause_rate_03", "pause_rate_06", "pause_rate_10", "mean_gap", "max_gap", "std_gap", "pause_time_ratio",
                "articulation_rate", "word_dur_mean", "word_dur_std", "prob_std", "prob_q10", "low_prob_ratio_05",
                "low_prob_ratio_03", "lead_silence", "trail_silence"]


def fluency_features(words_json, audio_duration):
    """Pause / rate / confidence statistics from Whisper word timestamps."""
    try:
        words = [w for w in json.loads(words_json) if w.get("start") is not None and w.get("end") is not None]
    except (TypeError, ValueError):
        words = []
    if len(words) < 3:
        return {k: np.nan for k in FLUENCY_KEYS}
    s = np.array([w["start"] for w in words], float)
    e = np.array([w["end"] for w in words], float)
    p = np.array([w["probability"] if w.get("probability") is not None else np.nan for w in words], float)
    p = p[np.isfinite(p)] if np.isfinite(p).any() else np.array([np.nan])
    gaps = np.clip(s[1:] - e[:-1], 0, None)
    span = max(e[-1] - s[0], 1.0)
    minutes = span / 60.0
    pause_time = gaps[gaps > 0.3].sum()
    dur = np.clip(e - s, 0, None)
    return {
        "pause_rate_03": (gaps > 0.3).sum() / minutes, "pause_rate_06": (gaps > 0.6).sum() / minutes,
        "pause_rate_10": (gaps > 1.0).sum() / minutes,
        "mean_gap": gaps.mean(), "max_gap": gaps.max(), "std_gap": gaps.std(),
        "pause_time_ratio": pause_time / span,
        "articulation_rate": len(words) / max(span - pause_time, 1.0),
        "word_dur_mean": dur.mean(), "word_dur_std": dur.std(),
        "prob_std": np.nanstd(p), "prob_q10": np.nanpercentile(p, 10),
        "low_prob_ratio_05": np.nanmean(p < 0.5), "low_prob_ratio_03": np.nanmean(p < 0.3),
        "lead_silence": s[0],
        "trail_silence": max(audio_duration - e[-1], 0.0) if pd.notna(audio_duration) else np.nan,
    }


def build_basic_features(df):
    out = df[ASR_RAW_COLS].apply(pd.to_numeric, errors="coerce").copy()
    out["words_per_audio_sec"] = out["word_count"] / out["audio_duration"].clip(lower=1)
    out["words_per_speech_sec"] = out["word_count"] / out["speech_span"].clip(lower=1)
    out["chars_per_word"] = out["char_count"] / out["word_count"].clip(lower=1)
    out["speech_span_ratio"] = out["speech_span"] / out["audio_duration"].clip(lower=1)
    ling = pd.DataFrame([linguistic_features(t) for t in df["transcript"]], index=df.index)
    return pd.concat([out, ling], axis=1)


def build_fluency_features(df):
    return pd.DataFrame([fluency_features(w, d) for w, d in zip(df["words_json"], df["audio_duration"])], index=df.index)


basic_tr, basic_te = build_basic_features(train_asr), build_basic_features(test_asr)
flu_tr, flu_te = build_fluency_features(train_asr), build_fluency_features(test_asr)
BASE_NUM_COLS = basic_tr.columns.tolist()     # the v1 numeric feature set
print(f"basic features: {len(BASE_NUM_COLS)} | fluency features: {flu_tr.shape[1]}")

In [ ]:
# ---- GPT-2 surprise features + spaCy syntax features (cached, keyed by transcript tag) ----
import torch.nn.functional as F_nn

LM_KEYS = ["lm_nll_mean", "lm_nll_median", "lm_nll_std", "lm_nll_q90", "lm_tok_gt6", "lm_tok_gt9",
           "lm_sent_mean", "lm_sent_max", "lm_sent_std", "lm_n_sent"]


@torch.no_grad()
def lm_features(texts):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(LM_NAME)
    tok.pad_token = tok.eos_token
    lm = AutoModelForCausalLM.from_pretrained(LM_NAME).to(device).eval()
    rows = []
    for text in tqdm(texts, desc="LM surprise"):
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(text)) if len(s.split()) >= 2]
        if not sents:
            rows.append({k: np.nan for k in LM_KEYS}); continue
        enc = tok([tok.bos_token + s for s in sents], return_tensors="pt", padding=True, truncation=True,
                  max_length=128).to(device)
        logits = lm(**enc).logits[:, :-1].float()
        mask = enc["attention_mask"][:, 1:].bool()
        nll = F_nn.cross_entropy(logits.transpose(1, 2), enc["input_ids"][:, 1:], reduction="none")
        tok_nll = nll[mask].cpu().numpy()
        sent_nll = ((nll * mask).sum(1) / mask.sum(1)).cpu().numpy()
        rows.append({"lm_nll_mean": tok_nll.mean(), "lm_nll_median": np.median(tok_nll), "lm_nll_std": tok_nll.std(),
                     "lm_nll_q90": np.percentile(tok_nll, 90), "lm_tok_gt6": (tok_nll > 6).mean(),
                     "lm_tok_gt9": (tok_nll > 9).mean(), "lm_sent_mean": sent_nll.mean(),
                     "lm_sent_max": sent_nll.max(), "lm_sent_std": sent_nll.std(), "lm_n_sent": len(sents)})
    del lm
    free_gpu()
    return pd.DataFrame(rows)


def load_spacy():
    try:
        import importlib, spacy
        try:
            return spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
        except OSError:
            import subprocess
            subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True, capture_output=True)
            importlib.invalidate_caches()
            return spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
    except Exception as err:  # noqa: BLE001
        print("spaCy unavailable -> syntax features skipped:", repr(err))
        return None


CLAUSE_DEPS = {"advcl", "ccomp", "xcomp", "relcl", "acl", "csubj"}
POS_TRACKED = ["NOUN", "VERB", "AUX", "ADJ", "ADV", "PRON", "ADP", "DET", "CCONJ", "SCONJ"]
VERB_TAGS = {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "MD"}


def token_depth(tok):
    depth = 0
    while tok.head.i != tok.i and depth < 50:      # compare indices: spaCy builds a new Token object on every access
        tok, depth = tok.head, depth + 1
    return depth


def syntax_features(nlp, texts):
    rows = []
    for doc in tqdm(nlp.pipe([str(t) for t in texts], batch_size=32), total=len(texts), desc="spaCy syntax"):
        sents = list(doc.sents)
        content = [t for t in doc if not t.is_punct and not t.is_space]
        n_tok, n_sent = max(len(content), 1), max(len(sents), 1)
        no_subj_or_verb = sum(
            (not any(t.dep_ in {"nsubj", "nsubjpass", "expl", "csubj"} for t in s)) or s.root.pos_ not in {"VERB", "AUX"}
            for s in sents)
        depths = [token_depth(t) for t in content] or [0]
        lens = [len([t for t in s if not t.is_punct]) for s in sents] or [0]
        row = {"syn_frag_ratio": no_subj_or_verb / n_sent, "syn_depth_mean": float(np.mean(depths)),
               "syn_depth_max": float(np.max(depths)),
               "syn_clause_rate": sum(t.dep_ in CLAUSE_DEPS for t in content) / n_sent,
               "syn_sconj_rate": sum(t.pos_ == "SCONJ" for t in content) / n_tok,
               "syn_passive_rate": sum(t.dep_ in {"nsubjpass", "auxpass"} for t in content) / n_sent,
               "syn_sent_len_mean": float(np.mean(lens)), "syn_sent_len_std": float(np.std(lens)),
               "syn_modal_rate": sum(t.tag_ == "MD" for t in content) / n_tok,
               "syn_verb_form_variety": len({t.tag_ for t in content if t.tag_ in VERB_TAGS})}
        for pos in POS_TRACKED:
            row[f"syn_pos_{pos.lower()}"] = sum(t.pos_ == pos for t in content) / n_tok
        rows.append(row)
    return pd.DataFrame(rows)


all_texts = train_asr["transcript"].tolist() + test_asr["transcript"].tolist()
TEXTFEAT_PATH = CACHE_DIR / f"text_features_{TAG}.csv"
if TEXTFEAT_PATH.exists():
    text_feats = pd.read_csv(TEXTFEAT_PATH)
    print("Loaded cached LM/syntax features:", text_feats.shape)
else:
    parts = [lm_features(all_texts)]
    nlp = load_spacy()
    if nlp is not None:
        parts.append(syntax_features(nlp, all_texts))
    text_feats = pd.concat(parts, axis=1)
    text_feats.to_csv(TEXTFEAT_PATH, index=False)
    print("Computed LM/syntax features:", text_feats.shape)

In [ ]:
# ---- v4: ELECTRA replaced-token detector as a grammar-anomaly signal (cached, keyed by transcript tag) ----
# ELECTRA's discriminator was pre-trained to spot tokens that do not belong in their context (a wrong word form, a
# misplaced function word, a repetition...). Its per-token "replaced" probability is therefore a direct,
# interpretable measure of local ungrammaticality that needs no grammar-error corpus.
EL_KEYS = ["el_p_mean", "el_p_q90", "el_p_max", "el_frac_gt50", "el_frac_gt20", "el_logit_mean", "el_logit_std"]


@torch.no_grad()
def electra_features(texts, name=ELECTRA_NAME, batch_size=16):
    from transformers import AutoTokenizer, ElectraForPreTraining
    tok = AutoTokenizer.from_pretrained(name)
    model = ElectraForPreTraining.from_pretrained(name).to(device).eval()
    safe = [t if str(t).strip() else "." for t in texts]
    rows = [None] * len(safe)
    order = np.argsort([len(t) for t in safe])
    for s in tqdm(range(0, len(safe), batch_size), desc="ELECTRA replaced-token scores"):
        idx = order[s:s + batch_size]
        enc = tok([safe[i] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
        logits = model(**enc).logits.float().cpu().numpy()                    # [B, T]; high = token looks out of place
        mask = enc["attention_mask"].cpu().numpy().astype(bool)
        lengths = mask.sum(1)
        for b, i in enumerate(idx):
            valid = mask[b].copy()
            valid[0] = False                                                  # [CLS]
            valid[lengths[b] - 1] = False                                     # [SEP]
            lg = logits[b][valid]
            if lg.size == 0:
                rows[i] = {k: np.nan for k in EL_KEYS}
                continue
            p = 1.0 / (1.0 + np.exp(-lg))
            rows[i] = {"el_p_mean": p.mean(), "el_p_q90": np.percentile(p, 90), "el_p_max": p.max(),
                       "el_frac_gt50": (p > 0.5).mean(), "el_frac_gt20": (p > 0.2).mean(),
                       "el_logit_mean": lg.mean(), "el_logit_std": lg.std()}
    del model
    free_gpu()
    return pd.DataFrame(rows)


electra_feats = None
if USE_ELECTRA:
    ELECTRA_PATH = CACHE_DIR / f"electra_features_{TAG}.csv"
    try:
        if ELECTRA_PATH.exists():
            electra_feats = pd.read_csv(ELECTRA_PATH)
            print("Loaded cached ELECTRA features:", electra_feats.shape)
        else:
            electra_feats = electra_features(all_texts)
            electra_feats.to_csv(ELECTRA_PATH, index=False)
            print("Computed ELECTRA features:", electra_feats.shape)
    except Exception as err:  # noqa: BLE001  optional feature family: never block the run
        print("[skipped optional ELECTRA features]", repr(err))
        electra_feats = None
        free_gpu()

In [ ]:
# ---- assemble feature tables; impute with TRAIN (rubric-valid) medians only ----
lm_syn_tr, lm_syn_te = text_feats.iloc[:N_TRAIN].reset_index(drop=True), text_feats.iloc[N_TRAIN:].reset_index(drop=True)
parts_tr = [basic_tr.reset_index(drop=True), flu_tr.reset_index(drop=True), lm_syn_tr]
parts_te = [basic_te.reset_index(drop=True), flu_te.reset_index(drop=True), lm_syn_te]
if electra_feats is not None:                                             # v4: ELECTRA anomaly features
    parts_tr.append(electra_feats.iloc[:N_TRAIN].reset_index(drop=True))
    parts_te.append(electra_feats.iloc[N_TRAIN:].reset_index(drop=True))
F_train, F_test = pd.concat(parts_tr, axis=1), pd.concat(parts_te, axis=1)
F_train, F_test = [f.replace([np.inf, -np.inf], np.nan) for f in (F_train, F_test)]
medians = F_train.iloc[idx_nz].median().fillna(0.0)
F_train, F_test = F_train.fillna(medians).astype(float), F_test.fillna(medians).astype(float)
ALL_NUM_COLS = F_train.columns.tolist()
print(f"{len(BASE_NUM_COLS)} basic + {len(ALL_NUM_COLS) - len(BASE_NUM_COLS)} new features = {len(ALL_NUM_COLS)} total")

## 5. Frozen audio embeddings — *every layer*, not just the last

v1 used only the last WavLM layer. In self-supervised speech models the last layer is tuned for the pre-training objective; intermediate layers usually carry more phonetic / lexical information, which is what grammar scoring needs. So for each backbone I store, for **every hidden layer**, the time-wise **mean and standard deviation** (std captures variability such as prosody and hesitation).
Clips are cut into chunks (15 s for WavLM, 30 s for Whisper) and chunk statistics are combined exactly (duration-weighted mean, pooled variance). Nothing is fine-tuned, and each backbone is cached as a float16 array `[clips, layers, 2·dim]`.

Backbones: **WavLM-base-plus** (v1's model), **WavLM-large**, the **Whisper** encoders (small / medium / large-v3; trained on 680k h of transcribed speech, so strongly linguistic) and, new in v4, **HuBERT-large** (self-supervised) and **wav2vec2-large-960h-lv60-self** (fine-tuned for ASR, so its upper layers are lexical). Backbones marked `optional` are skipped if they fail to load.

In [ ]:
from transformers import AutoFeatureExtractor, AutoModel, AutoTokenizer


class SSLExtractor:
    """WavLM / HuBERT / wav2vec2: per-layer time statistics of one audio chunk."""

    def __init__(self, name):
        try:
            self.fe = AutoFeatureExtractor.from_pretrained(name)
        except Exception:  # noqa: BLE001  some checkpoints ship no preprocessor_config.json
            from transformers import Wav2Vec2FeatureExtractor
            self.fe = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=TARGET_SR, padding_value=0.0,
                                               do_normalize=True, return_attention_mask=False)
        self.model = AutoModel.from_pretrained(name).to(device).eval()

    @torch.no_grad()
    def chunk_stats(self, chunk):
        x = self.fe(chunk, sampling_rate=TARGET_SR, return_tensors="pt").input_values.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            hidden = self.model(x, output_hidden_states=True).hidden_states
        H = torch.stack([h[0].float() for h in hidden])                   # [layers, T, D]
        return H.mean(1), H.var(1, unbiased=False), H.shape[1]


class WhisperEncExtractor:
    """Whisper encoder only. Padding frames (Whisper pads every chunk to 30 s) are excluded from pooling."""

    def __init__(self, name):
        from transformers import WhisperFeatureExtractor, WhisperModel
        self.fe = WhisperFeatureExtractor.from_pretrained(name)
        full = WhisperModel.from_pretrained(name, torch_dtype=torch.float16, low_cpu_mem_usage=True)
        self.model = full.encoder.half().to(device).eval()
        del full

    @torch.no_grad()
    def chunk_stats(self, chunk):
        feats = self.fe(chunk, sampling_rate=TARGET_SR, return_tensors="pt").input_features.to(device, torch.float16)
        hidden = self.model(feats, output_hidden_states=True).hidden_states
        n_valid = max(1, min(1500, int(math.ceil(len(chunk) / 320))))     # 50 encoder frames per second
        H = torch.stack([h[0, :n_valid].float() for h in hidden])
        return H.mean(1), H.var(1, unbiased=False), H.shape[1]


def pool_chunks(stats):
    """Combine per-chunk (mean, var, n_frames) into one [layers, 2*dim] vector (exact pooled mean and std)."""
    w = np.array([s[2] for s in stats], dtype=np.float64)
    w /= w.sum()
    means = np.stack([s[0].cpu().numpy() for s in stats]).astype(np.float64)
    vars_ = np.stack([s[1].cpu().numpy() for s in stats]).astype(np.float64)
    m = np.tensordot(w, means, axes=1)
    v = np.tensordot(w, vars_ + means ** 2, axes=1) - m ** 2
    return np.concatenate([m, np.sqrt(np.clip(v, 0, None))], axis=-1).astype(np.float32)


def extract_audio_layers(key, cfg):
    final, part = CACHE_DIR / f"audio_{key}.npy", CACHE_DIR / f"audio_{key}.partial.npy"
    if final.exists():
        return np.load(final).astype(np.float32)
    paths = [TRAIN_DIR / f for f in train_df["filename"]] + [TEST_DIR / f for f in test_df["filename"]]
    rows = list(np.load(part)) if part.exists() else []
    extractor = (WhisperEncExtractor if cfg["kind"] == "whisper" else SSLExtractor)(cfg["name"])
    chunk = int(cfg["chunk_sec"] * TARGET_SR)
    for i in tqdm(range(len(rows), len(paths)), desc=key):
        audio = load_audio_16k(paths[i])
        pieces = [audio[s:s + chunk] for s in range(0, len(audio), chunk)]
        pieces = [p for p in pieces if len(p) >= 3200] or [np.zeros(TARGET_SR, np.float32)]
        rows.append(pool_chunks([extractor.chunk_stats(p) for p in pieces]))
        if (i + 1) % 100 == 0:
            np.save(part, np.stack(rows).astype(np.float16))
    arr = np.stack(rows).astype(np.float16)
    np.save(final, arr)
    part.unlink(missing_ok=True)
    del extractor
    free_gpu()
    return arr.astype(np.float32)


AUDIO = {}          # key -> (train_array, test_array), each [clips, layers, 2*dim]
for key, cfg in AUDIO_BACKBONES.items():
    try:
        arr = extract_audio_layers(key, cfg)
        assert np.isfinite(arr).all(), f"non-finite values in {key}"
    except Exception as err:  # noqa: BLE001
        if not cfg.get("optional"):
            raise
        print(f"[skipped optional audio backbone {key}] {err!r}")
        free_gpu()
        continue
    AUDIO[key] = (arr[:N_TRAIN], arr[N_TRAIN:])
    print(f"{key:<20} train {AUDIO[key][0].shape}  test {AUDIO[key][1].shape}")

### 5b. Whisper *decoder* view of the transcript  *(new in v5)*

The encoder views above never see the transcript. Here Whisper's **decoder** is run over the ASR transcript (teacher-forced), cross-attending to the audio, and the hidden state of every decoder layer is mean/std-pooled over the transcript tokens — the same `[clips, layers, 2·dim]` format as the audio backbones, so it goes through the same layer-probing heads.
A clip is cut into 30 s windows; each word is assigned to the window it starts in (from Whisper's word timestamps) so every window's decoder sees the audio that contains its words. Decoder states depend on the transcripts, so the cache is keyed by the transcript tag. The backbone is optional: if it fails it is skipped with a message.

In [ ]:
class WhisperDecExtractor:
    """Whisper decoder hidden states over a transcript piece (teacher forcing), conditioned on the matching 30 s of audio."""

    def __init__(self, name, multilingual=False):
        from transformers import WhisperFeatureExtractor, WhisperModel, WhisperTokenizer
        self.fe = WhisperFeatureExtractor.from_pretrained(name)
        self.tok = WhisperTokenizer.from_pretrained(name)
        self.model = WhisperModel.from_pretrained(name, torch_dtype=torch.float16, low_cpu_mem_usage=True).to(device).eval()
        special = ["<|startoftranscript|>"] + (["<|en|>", "<|transcribe|>"] if multilingual else []) + ["<|notimestamps|>"]
        self.prefix = self.tok.convert_tokens_to_ids(special)
        assert all(isinstance(i, int) and i != self.tok.unk_token_id for i in self.prefix), f"missing special tokens {special} -> {self.prefix}"
        self.max_text = self.model.config.max_target_positions - len(self.prefix) - 1
        self.shape = (self.model.config.decoder_layers + 1, self.model.config.d_model)

    @torch.no_grad()
    def chunk_stats(self, audio_chunk, text):
        ids = self.tok.encode(" " + text.strip(), add_special_tokens=False)[: self.max_text]
        if not ids:
            return None
        feats = self.fe(audio_chunk, sampling_rate=TARGET_SR, return_tensors="pt").input_features.to(device, torch.float16)
        dec = torch.tensor([self.prefix + ids], device=device)
        out = self.model(input_features=feats, decoder_input_ids=dec, output_hidden_states=True)
        H = torch.stack([h[0, len(self.prefix):].float() for h in out.decoder_hidden_states])      # [layers, T, D]
        return H.mean(1), H.var(1, unbiased=False), H.shape[1]


def words_by_chunk(words_json, chunk_sec, n_chunks):
    """Split a clip's transcript into one string per audio window, using each word's start time."""
    try:
        words = json.loads(words_json)
    except (TypeError, ValueError):
        words = []
    buckets = [[] for _ in range(n_chunks)]
    for w in words:
        if w.get("start") is None or not str(w.get("word", "")).strip():
            continue
        buckets[min(n_chunks - 1, int(w["start"] // chunk_sec))].append(str(w["word"]).strip())
    return [" ".join(b) for b in buckets]


def extract_decoder_layers(key, cfg):
    final = CACHE_DIR / f"audio_{key}_{TAG}.npy"
    part = CACHE_DIR / f"audio_{key}_{TAG}.partial.npy"
    if final.exists():
        return np.load(final).astype(np.float32)
    paths = [TRAIN_DIR / f for f in train_df["filename"]] + [TEST_DIR / f for f in test_df["filename"]]
    wjs = train_asr["words_json"].tolist() + test_asr["words_json"].tolist()
    rows = list(np.load(part)) if part.exists() else []
    extractor = WhisperDecExtractor(cfg["name"], cfg.get("multilingual", False))
    chunk = int(cfg["chunk_sec"] * TARGET_SR)
    for i in tqdm(range(len(rows), len(paths)), desc=key):
        audio = load_audio_16k(paths[i])
        n_chunks = max(1, math.ceil(len(audio) / chunk))
        texts = words_by_chunk(wjs[i], cfg["chunk_sec"], n_chunks)
        stats = []
        for c, t in enumerate(texts):
            piece = audio[c * chunk:(c + 1) * chunk]
            if len(piece) < 3200:
                continue
            s = extractor.chunk_stats(piece, t)
            if s is not None:
                stats.append(s)
        rows.append(pool_chunks(stats) if stats else np.zeros((extractor.shape[0], 2 * extractor.shape[1]), np.float32))
        if (i + 1) % 100 == 0:
            np.save(part, np.stack(rows).astype(np.float16))
    arr = np.stack(rows).astype(np.float16)
    np.save(final, arr)
    part.unlink(missing_ok=True)
    del extractor
    free_gpu()
    return arr.astype(np.float32)


for key, cfg in DECODER_BACKBONES.items():
    try:
        arr = extract_decoder_layers(key, cfg)
        assert np.isfinite(arr).all(), f"non-finite values in {key}"
    except Exception as err:  # noqa: BLE001
        if not cfg.get("optional"):
            raise
        print(f"[skipped optional decoder backbone {key}] {err!r}")
        free_gpu()
        continue
    AUDIO[key] = (arr[:N_TRAIN], arr[N_TRAIN:])          # same [clips, layers, 2*dim] format -> probed like an audio backbone
    print(f"{key:<20} train {AUDIO[key][0].shape}  test {AUDIO[key][1].shape}")

## 6. Transcript embeddings

* **MPNet** (as in v1) and, new, **bge-large** sentence embeddings → SVR on embedding + numeric features.
* **Layer-wise features** of frozen language models, probed with the same machinery as audio: **RoBERTa-large** (written-text MLM; middle layers encode syntax),
  **Qwen2.5-1.5B** (decoder LLM; the first "attention-sink" token is excluded from pooling) and **DeBERTa-v3-large** (ELECTRA-style replaced-token pre-training → sensitive to words that do not fit).
All are frozen and cached under the transcript tag; optional ones are skipped if they fail.

In [ ]:
@torch.no_grad()
def extract_text_layers(cfg, texts, max_len=512):
    """Frozen encoder / LLM: per-layer mean and std over the (non-padding) tokens of each transcript."""
    name, batch_size = cfg["name"], cfg.get("batch", 16)
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token or tok.unk_token
    tok.padding_side = "right"
    dtype = torch.float16 if cfg.get("dtype", "fp16") == "fp16" else torch.float32
    model = AutoModel.from_pretrained(name, torch_dtype=dtype).to(device).eval()
    n_layers = model.config.num_hidden_layers + 1
    safe = [t if str(t).strip() else "." for t in texts]                 # a completely empty batch would crash the model
    out = np.zeros((len(safe), n_layers, 2 * model.config.hidden_size), dtype=np.float32)
    order = np.argsort([len(t) for t in safe])
    for s in tqdm(range(0, len(safe), batch_size), desc=name):
        idx = order[s:s + batch_size]
        enc = tok([safe[i] for i in idx], return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(device)
        hidden = torch.stack(model(**enc, output_hidden_states=True).hidden_states).float()   # [L, B, T, D]
        mask = enc["attention_mask"].clone()
        if cfg.get("skip_first"):          # decoder LLMs: the first token is an "attention sink" with huge, constant activations
            mask[:, 0] = 0
        m = mask[None, :, :, None].float()
        cnt = m.sum(2).clamp(min=1.0)
        mean = (hidden * m).sum(2) / cnt
        var = (((hidden - mean[:, :, None]) ** 2) * m).sum(2) / cnt
        out[idx] = torch.cat([mean, var.sqrt()], dim=-1).permute(1, 0, 2).cpu().numpy()
        del hidden
    del model
    free_gpu()
    return out


TEXT_LAYERS = {}
for key, cfg in TEXT_BACKBONES.items():
    path = CACHE_DIR / f"text_{key}_{TAG}.npy"
    try:
        if path.exists():
            arr = np.load(path).astype(np.float32)
        else:
            arr = extract_text_layers(cfg, all_texts)
            assert np.isfinite(arr).all(), f"non-finite values in {key} (try dtype='fp32' for this backbone)"
            np.save(path, arr)
    except Exception as err:  # noqa: BLE001
        if not cfg.get("optional"):
            raise
        print(f"[skipped optional text backbone {key}] {err!r}")
        free_gpu()
        continue
    TEXT_LAYERS[key] = (arr[:N_TRAIN], arr[N_TRAIN:])
    print(f"{key:<20} train {TEXT_LAYERS[key][0].shape}")


def encode_sentences(model_name, texts):
    from sentence_transformers import SentenceTransformer
    enc = SentenceTransformer(model_name, device="cuda")
    arr = enc.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    del enc
    free_gpu()
    return arr


mp_path = CACHE_DIR / f"mpnet_{TAG}.npy"
if mp_path.exists():
    mpnet_all = np.load(mp_path)
else:
    mpnet_all = encode_sentences(SENTENCE_ENCODER, all_texts)
    np.save(mp_path, mpnet_all)
mpnet_train, mpnet_test = mpnet_all[:N_TRAIN], mpnet_all[N_TRAIN:]
print("mpnet", mpnet_train.shape)

EXTRA_SENT = {}          # v4: further sentence encoders -> extra, differently-biased SVR models
for key, model_name in EXTRA_SENTENCE_ENCODERS.items():
    path = CACHE_DIR / f"{key}_{TAG}.npy"
    try:
        arr = np.load(path) if path.exists() else encode_sentences(model_name, all_texts)
        if not path.exists():
            np.save(path, arr)
    except Exception as err:  # noqa: BLE001
        print(f"[skipped optional sentence encoder {key}] {err!r}")
        free_gpu()
        continue
    EXTRA_SENT[key] = (arr[:N_TRAIN], arr[N_TRAIN:])
    print(key, arr[:N_TRAIN].shape)

## 7. Modelling infrastructure

**Validation.** `N_REPEATS` repeats of five stratified folds over the rubric-valid clips, each repeat with a different seed. With `CV_MODE="speaker"` (v5 default) the folds are `StratifiedGroupKFold` over **inferred speaker groups**:
early WavLM layers mostly encode voice, nearly every training clip has a near-identical voice elsewhere in training (usually its neighbours in the file numbering) while no test clip does, so training clips whose voice similarity exceeds the 99.5th percentile of the *test→train* similarities are linked and the connected components are the groups.
`CV_MODE="random"` reproduces v1/v4's folds (and is what the leakage check in Section 12a compares against).
Every model is evaluated the same way: fit on four folds, predict the fifth (→ out-of-fold, OOF) *and* predict the test set. OOF predictions are averaged over repeats, and so are the
5×`N_REPEATS` test predictions, so the stacker in Section 9 learns from inputs with exactly the character of the test inputs it will blend.

**Layer probing without leakage.** For every training set a `LayerRanking` scores each (layer, pooling) by the *exact leave-one-out error of ridge* (closed form from one eigendecomposition),
and four heads then reuse that single ranking — all selection of layers, `k`, `C`, `γ`, `α` happens **inside** `fit()` on the training fold only:

| head | uses | selects by |
|---|---|---|
| `svr` | top-k layers concatenated, RBF-SVR | 3-fold inner CV over (k, C) |
| `ridge` | same layers, linear ridge | inner CV over (k, α) |
| `krr` | same layers, RBF kernel ridge | **exact LOO** over (k, γ, α) |
| `lstack` | **all** layers: one ridge each, non-negative weights on their exact LOO predictions | LOO |

In [ ]:
from scipy import sparse
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import KFold, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR


def rmse(y_true, pred):
    return float(np.sqrt(np.mean((np.asarray(y_true, float) - np.asarray(pred, float)) ** 2)))


def evaluate(y_true, pred):
    y_true, pred = np.asarray(y_true, float), np.asarray(pred, float)
    rnd = lambda v: np.floor(np.clip(v, 1, 5) + 0.5).astype(int)
    return {"rmse": rmse(y_true, pred), "mae": float(np.mean(np.abs(y_true - pred))),
            "pearson": float(pearsonr(y_true, pred)[0]), "spearman": float(spearmanr(y_true, pred)[0]),
            "qwk": float(cohen_kappa_score(rnd(y_true), rnd(pred), weights="quadratic"))}


def mean_metrics(y_true, preds):
    """Average the metrics of several prediction vectors (one per CV repeat) - an honest, un-smoothed headline number."""
    rows = [evaluate(y_true, p) for p in preds]
    return {k: float(np.mean([r[k] for r in rows])) for k in rows[0]}


def take(X, idx):
    return X.iloc[idx] if isinstance(X, pd.DataFrame) else X[idx]


# ---------------------------------------------------------------- speaker groups and folds (v5)
y = train_df["label"].values[idx_nz].astype(float)
strata = np.digitize(y, [2.25, 3.25, 4.25], right=True)      # same bins as v1
B_TR, B_TE = BATCH[:N_TRAIN][idx_nz], BATCH[N_TRAIN:]        # recording batch of every rubric-valid train clip / test clip


def file_number(name):
    return int(re.search(r"(\d+)", str(name)).group(1))


def build_speaker_groups(percentile=SPEAKER_PERCENTILE, layers=VOICE_LAYERS):
    """Infer speaker groups of ALL training clips from early SSL layers (they encode the voice, not the content).

    Voice vectors = per-layer time mean and std of the early layers, standardised over train+test clips and L2-normalised.
    Test speakers never occur in training, so the distribution of test->train nearest-neighbour similarity describes
    *different* speakers; training pairs above its `percentile` are linked, connected components are the groups."""
    from scipy.sparse.csgraph import connected_components
    key = next((k for k in ("wavlm_large", "wavlm_base_plus") if k in AUDIO), next(iter(AUDIO)))
    A = np.concatenate([AUDIO[key][0], AUDIO[key][1]], axis=0)                # [train+test, layers, 2*dim] (mean | std)
    D = A.shape[2] // 2
    layers = [l for l in layers if l < A.shape[1]]
    V = np.hstack([A[:, l, :D] for l in layers] + [A[:, l, D:] for l in layers]).astype(np.float64)
    V = (V - V.mean(0)) / (V.std(0) + 1e-6)
    V /= np.linalg.norm(V, axis=1, keepdims=True)
    sim = V @ V.T
    np.fill_diagonal(sim, -1.0)
    s_tr = sim[:N_TRAIN, :N_TRAIN]
    nn_train, nn_test = s_tr.max(1), sim[N_TRAIN:, :N_TRAIN].max(1)
    tau = float(np.percentile(nn_test, percentile))
    n_groups, groups = connected_components(s_tr > tau, directed=False)
    return groups, {"backbone": key, "layers": layers, "tau": tau, "nn_train": nn_train, "nn_test": nn_test, "n_groups": int(n_groups)}


SPK_GROUPS, SPK_INFO = build_speaker_groups()
_sizes = np.bincount(SPK_GROUPS)
_multi = [g for g in range(len(_sizes)) if 1 < _sizes[g] < 10 and ((SPK_GROUPS == g) & (lab_all > 0)).sum() > 1]
_within = float(np.mean([lab_all[(SPK_GROUPS == g) & (lab_all > 0)].std() for g in _multi])) if _multi else float("nan")
print(f"speaker groups from {SPK_INFO['backbone']} layers {SPK_INFO['layers']}: {SPK_INFO['n_groups']} groups | sizes 1/2/3/4+: "
      f"{(_sizes == 1).sum()}/{(_sizes == 2).sum()}/{(_sizes == 3).sum()}/{(_sizes >= 4).sum()} (largest {_sizes.max()}) | "
      f"grade std within a group {_within:.2f} vs {lab_all[lab_all > 0].std():.2f} overall")
print(f"train clips with a near-identical voice elsewhere in train: {(SPK_INFO['nn_train'] > SPK_INFO['tau']).mean():.0%} | test clips: "
      f"{(SPK_INFO['nn_test'] > SPK_INFO['tau']).mean():.0%} (should be ~0-1%)")
assert _sizes.max() < 0.2 * N_TRAIN, "one speaker group holds >20% of the clips - the voice threshold is too loose; check VOICE_LAYERS / SPEAKER_PERCENTILE"
groups_nz = SPK_GROUPS[idx_nz]

plt.figure(figsize=(6.5, 3))
_edges = np.linspace(min(SPK_INFO["nn_train"].min(), SPK_INFO["nn_test"].min()), 1.0, 60)
plt.hist(SPK_INFO["nn_train"], _edges, alpha=0.6, density=True, label="train clip -> nearest train clip")
plt.hist(SPK_INFO["nn_test"], _edges, alpha=0.6, density=True, label="test clip -> nearest train clip")
plt.axvline(SPK_INFO["tau"], color="r", ls="--")
plt.xlabel("voice similarity"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()


def make_folds(mode, seed=SEED):
    if mode == "speaker":
        splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return list(splitter.split(np.zeros(len(y)), strata, groups_nz))
    if mode == "group":
        groups = train_df["filename"].iloc[idx_nz].map(file_number).values // GROUP_SIZE
        splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        return list(splitter.split(np.zeros(len(y)), strata, groups))
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    return list(splitter.split(np.zeros(len(y)), strata))


FOLD_SETS = [make_folds(CV_MODE, SEED + 101 * r) for r in range(N_REPEATS)]   # repeat 0 = v1's folds when CV_MODE="random"
folds = FOLD_SETS[0]
fold_id = np.zeros(len(y), dtype=int)
for k, (_, va) in enumerate(folds):
    fold_id[va] = k
if CV_MODE == "speaker":                                   # no speaker group may straddle a train/validation split
    for fs in FOLD_SETS:
        for tr, va in fs:
            assert not (set(groups_nz[tr]) & set(groups_nz[va])), "speaker group split across folds"
def _fold_of(fs):
    f = np.zeros(len(y), dtype=np.int8)
    for k, (_, va) in enumerate(fs):
        f[va] = k
    return f


_fold_ids = np.stack([_fold_of(fs) for fs in FOLD_SETS])
FOLD_HASH = hashlib.md5(_fold_ids.tobytes()).hexdigest()[:8]
CV_ID = "random" if CV_MODE == "random" else f"{CV_MODE}-{FOLD_HASH}"   # part of every prediction-cache key (folds changed -> new cache)


def testmix_rmse(y_true, preds):
    """RMSE re-weighted so each recording batch counts as often as it does in the TEST set (batch shares are known from
    the test durations; no labels involved). Mean over the prediction vectors in `preds` (one per CV repeat)."""
    y_true = np.asarray(y_true, float)
    out = []
    for p in preds:
        p = np.asarray(p, float)
        mse = np.array([np.mean((y_true[B_TR == b] - p[B_TR == b]) ** 2) if (B_TR == b).any() else np.nan for b in range(4)])
        w = np.where(np.isnan(mse), 0.0, TEST_SHARE)
        out.append(float(np.sqrt(np.nansum(w * mse) / w.sum())))
    return float(np.mean(out))


def run_cv(make_model, X, y, X_test, fold_sets=None):
    """Repeated K-fold. Returns (OOF averaged over repeats, test prediction averaged over ALL fold models, OOF per repeat).

    Every clip is predicted by models that never saw it in each repeat; averaging the repeats reduces the variance that
    comes from one particular random split (the main noise source with ~730 clips)."""
    fold_sets = FOLD_SETS if fold_sets is None else fold_sets
    oof_rep, test_preds = np.zeros((len(fold_sets), len(y))), []
    for r, fs in enumerate(fold_sets):
        for tr, va in fs:
            model = make_model().fit(take(X, tr), y[tr])
            oof_rep[r, va] = model.predict(take(X, va))
            test_preds.append(model.predict(X_test))
    return oof_rep.mean(0), np.mean(test_preds, axis=0), oof_rep


def run_cv_bundle(make_bundle, X, y, X_test, fold_sets=None):
    """Same as run_cv for an estimator whose predict() returns {head_name: predictions}; one fit serves every head."""
    fold_sets = FOLD_SETS if fold_sets is None else fold_sets
    oof_rep, test_preds = None, None
    for r, fs in enumerate(fold_sets):
        for tr, va in fs:
            model = make_bundle().fit(take(X, tr), y[tr])
            pv, pt = model.predict(take(X, va)), model.predict(X_test)
            if oof_rep is None:
                oof_rep = {h: np.zeros((len(fold_sets), len(y))) for h in pv}
                test_preds = {h: [] for h in pv}
            for h in pv:
                oof_rep[h][r, va] = pv[h]
                test_preds[h].append(pt[h])
    return {h: (oof_rep[h].mean(0), np.mean(test_preds[h], axis=0), oof_rep[h]) for h in oof_rep}


print(f"CV mode: {CV_MODE} ({CV_ID}) | {N_REPEATS} repeat(s) x {N_SPLITS} folds | {len(y)} rubric-valid clips | "
      f"fold sizes (repeat 0): {[len(va) for _, va in folds]}")
print(f"Label mean {y.mean():.3f}, std {y.std():.3f} -> predicting the mean gives RMSE ≈ {y.std():.3f}")

In [ ]:
# ---------------------------------------------------------------- layer probing (v4: one ranking, four heads)
ALPHA_MULTS = np.logspace(-2, 2.5, 10)      # ridge penalty = multiplier * n_features (features are standardised)


def ridge_loo(Z, y, alpha_mults=ALPHA_MULTS, return_pred=False):
    """Exact leave-one-out error of ridge regression (unpenalised intercept) for several penalties.

    One eigendecomposition of the n x n kernel matrix gives the hat matrix for every alpha, and the LOO residual is
    (y - y_hat) / (1 - h_ii).  Z must be column-centred.  Returns (best_rmse, best_alpha[, LOO predictions at best alpha]).
    """
    n, d = Z.shape
    yc = y - y.mean()
    w, Q = np.linalg.eigh(Z @ Z.T)
    w = np.clip(w, 0.0, None)
    Qty, Q2 = Q.T @ yc, Q ** 2
    best = (np.inf, None, None)
    for m in alpha_mults:
        a = m * d
        shrink = w / (w + a)
        resid = (yc - Q @ (shrink * Qty)) / (1.0 - (Q2 @ shrink + 1.0 / n))
        r = float(np.sqrt(np.mean(resid ** 2)))
        if r < best[0]:
            best = (r, a, (y - resid) if return_pred else None)
    return best if return_pred else best[:2]


def _standardise(Z):
    mu, sd = Z.mean(0), Z.std(0)
    sd[sd < 1e-8] = 1.0
    return mu, sd


def score_layers(X, y, return_preds=False):
    """LOO-ridge RMSE of every (layer, pooling) candidate. X: [n, layers, 2*dim] (first half mean, second half std)."""
    n, L, F = X.shape
    D = F // 2
    rows, preds = [], []
    for layer in range(L):
        for pool, sl in (("mean", slice(0, D)), ("mean+std", slice(0, F))):
            Z = X[:, layer, sl].astype(np.float64)
            mu, sd = _standardise(Z)
            r, a, p = ridge_loo((Z - mu) / sd, y, return_pred=True)
            rows.append((layer, pool, r, a))
            preds.append(p)
    table = pd.DataFrame(rows, columns=["layer", "pool", "loo_rmse", "alpha"])
    return (table, np.column_stack(preds)) if return_preds else table


class LayerRanking:
    """Leakage-free layer scoring on ONE training set, shared by all heads (computed once per fold, not once per head)."""

    def __init__(self, X, y):
        self.table, self.loo = score_layers(X, y, return_preds=True)
        self.table["cand"] = np.arange(len(self.table))
        self.best = (self.table.loc[self.table.groupby("layer")["loo_rmse"].idxmin()]
                     .sort_values("loo_rmse").reset_index(drop=True))
        self.ranked = [(int(r.layer), r.pool) for r in self.best.itertuples()]


class _Design:
    @staticmethod
    def _slice(X, layer, pool):
        D = X.shape[2] // 2
        return X[:, layer, :D] if pool == "mean" else X[:, layer, :]

    def _prepare(self, X, ranked, max_k):
        self.stats_ = {c: _standardise(self._slice(X, *c).astype(np.float64)) for c in ranked[:max_k]}

    def _design(self, X, selected):
        blocks = []
        for layer, pool in selected:
            mu, sd = self.stats_[(layer, pool)]
            blocks.append((self._slice(X, layer, pool).astype(np.float64) - mu) / sd)
        return np.hstack(blocks).astype(np.float32)


class _TopKHead(_Design):
    """Concatenate the k best layers, pick (k, hyper-parameter) by inner CV, fit the final estimator."""
    params = ()

    def __init__(self, k_grid=(1, 3, 6), inner_splits=3, seed=SEED):
        self.k_grid, self.inner_splits, self.seed = k_grid, inner_splits, seed

    def _make(self, p, n_features):
        raise NotImplementedError

    def fit(self, X, y, ranking):
        ranked = ranking.ranked
        max_k = min(max(self.k_grid), len(ranked))
        self._prepare(X, ranked, max_k)
        inner = list(KFold(self.inner_splits, shuffle=True, random_state=self.seed).split(np.zeros(len(y))))
        best = (np.inf, None, None)
        for k in sorted({min(k, max_k) for k in self.k_grid}):
            Z = self._design(X, ranked[:k])
            for p in self.params:
                pred = np.zeros(len(y))
                for tr, va in inner:
                    pred[va] = self._make(p, Z.shape[1]).fit(Z[tr], y[tr]).predict(Z[va])
                r = rmse(y, pred)
                if r < best[0]:
                    best = (r, k, p)
        _, self.k_, self.p_ = best
        self.selected_ = ranked[:self.k_]
        Z = self._design(X, self.selected_)
        self.est_ = self._make(self.p_, Z.shape[1]).fit(Z, y)
        return self

    def predict(self, X):
        return self.est_.predict(self._design(np.asarray(X), self.selected_))


class SVRHead(_TopKHead):
    params = (1.0, 3.0, 10.0)                       # C

    def _make(self, p, n_features):
        return SVR(C=p, epsilon=0.1, gamma="scale")


class RidgeHead(_TopKHead):
    params = (0.1, 0.5, 2.0, 10.0, 50.0)            # penalty multiplier (x n_features)

    def _make(self, p, n_features):
        return Ridge(alpha=p * n_features)


class KRRHead(_Design):
    """RBF kernel ridge on the k best layers. (k, gamma, alpha) are chosen by the EXACT leave-one-out error of kernel
    ridge (closed form from one eigendecomposition per gamma) - no inner CV loop, so it is cheap and uses all the data."""

    def __init__(self, k_grid=(1, 3, 6), g_grid=(0.25, 0.5, 1.0, 2.0), a_grid=tuple(np.logspace(-2.5, 0.5, 9))):
        self.k_grid, self.g_grid, self.a_grid = k_grid, g_grid, a_grid

    @staticmethod
    def _sqdist(A, B):
        d2 = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2.0 * A @ B.T
        return np.clip(d2, 0.0, None)

    def fit(self, X, y, ranking):
        ranked = ranking.ranked
        max_k = min(max(self.k_grid), len(ranked))
        self._prepare(X, ranked, max_k)
        n, ym = len(y), y.mean()
        yc = y - ym
        best = (np.inf, None, None, None)
        for k in sorted({min(k, max_k) for k in self.k_grid}):
            Z = self._design(X, ranked[:k]).astype(np.float64)
            d = Z.shape[1]
            D2 = self._sqdist(Z, Z)
            for g in self.g_grid:
                w, Q = np.linalg.eigh(np.exp(-g * D2 / d))
                w = np.clip(w, 0.0, None)
                Qty, Q2 = Q.T @ yc, Q ** 2
                for a in self.a_grid:
                    shrink = w / (w + a)
                    resid = (yc - Q @ (shrink * Qty)) / (1.0 - Q2 @ shrink)
                    r = float(np.sqrt(np.mean(resid ** 2)))
                    if r < best[0]:
                        best = (r, k, g, a)
        _, self.k_, self.g_, self.a_ = best
        self.selected_ = ranked[:self.k_]
        self.Z_ = self._design(X, self.selected_).astype(np.float64)
        self.d_ = self.Z_.shape[1]
        K = np.exp(-self.g_ * self._sqdist(self.Z_, self.Z_) / self.d_)
        self.coef_ = np.linalg.solve(K + self.a_ * np.eye(n), yc)
        self.ym_ = ym
        return self

    def predict(self, X):
        Zt = self._design(np.asarray(X), self.selected_).astype(np.float64)
        return np.exp(-self.g_ * self._sqdist(Zt, self.Z_) / self.d_) @ self.coef_ + self.ym_


class LayerStackHead(_Design):
    """Uses ALL layers: one ridge per layer (alpha from LOO), then a non-negative combination of their exact LOO predictions.

    The LOO predictions are genuinely out-of-sample, so the combination weights are learned without leakage. Where the
    top-k heads commit to a few layers, this head lets every informative layer vote - a different bias, hence diversity."""

    def __init__(self, top_n=12):
        self.top_n = top_n

    def fit(self, X, y, ranking):
        cand = ranking.table.set_index("cand")
        order = ranking.table.loc[ranking.table.groupby("layer")["loo_rmse"].idxmin()]      # best pooling per layer
        cols = order.sort_values("loo_rmse").head(self.top_n)["cand"].values
        stack = LinearRegression(positive=True).fit(ranking.loo[:, cols], y)
        self.members_, self.weights_ = [], []
        for c, w in zip(cols, stack.coef_):
            if w <= 1e-6:
                continue
            row = cand.loc[c]
            sl = (int(row["layer"]), row["pool"])
            mu, sd = _standardise(self._slice(X, *sl).astype(np.float64))
            Z = (self._slice(X, *sl).astype(np.float64) - mu) / sd
            self.members_.append((sl, mu, sd, Ridge(alpha=float(row["alpha"])).fit(Z, y)))
            self.weights_.append(float(w))
        self.intercept_ = float(stack.intercept_)          # no layer helps -> the model degrades gracefully to the mean
        return self

    def predict(self, X):
        X = np.asarray(X)
        out = np.full(len(X), self.intercept_, dtype=float)
        for ((layer, pool), mu, sd, rd), w in zip(self.members_, self.weights_):
            out += w * rd.predict((self._slice(X, layer, pool).astype(np.float64) - mu) / sd)
        return out


HEAD_CLASSES = {"svr": SVRHead, "ridge": RidgeHead, "krr": KRRHead, "lstack": LayerStackHead}


class ProbeBundle:
    """Rank layers once, then fit every requested head on that ranking; predict() returns {head: predictions}."""

    def __init__(self, heads=None):
        self.heads = tuple(PROBE_HEADS if heads is None else heads)

    def fit(self, X, y):
        X, y = np.asarray(X), np.asarray(y, float)
        self.ranking_ = LayerRanking(X, y)
        self.fitted_ = {h: HEAD_CLASSES[h]().fit(X, y, self.ranking_) for h in self.heads}
        return self

    def predict(self, X):
        return {h: m.predict(X) for h, m in self.fitted_.items()}

In [ ]:
# ---------------------------------------------------------------- text-side estimators (v1 recipes, self-tuning)
class TfidfRidge:
    """Word + character TF-IDF plus scaled numeric features -> Ridge; alpha chosen by inner CV inside fit()."""

    def __init__(self, num_cols, alphas=(0.3, 1.0, 3.0, 10.0), seed=SEED):
        self.num_cols, self.alphas, self.seed = list(num_cols), alphas, seed

    def _matrix(self, df, fit=False):
        texts = df["transcript"].values
        num = df[self.num_cols].values.astype(float)
        if fit:
            self.word = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2), min_df=2,
                                        max_df=0.98, max_features=25000, sublinear_tf=True)
            self.char = TfidfVectorizer(analyzer="char_wb", lowercase=True, ngram_range=(3, 5), min_df=2,
                                        max_features=30000, sublinear_tf=True)
            self.scaler = StandardScaler()
            blocks = [self.word.fit_transform(texts), self.char.fit_transform(texts), self.scaler.fit_transform(num)]
        else:
            blocks = [self.word.transform(texts), self.char.transform(texts), self.scaler.transform(num)]
        return sparse.hstack([sparse.csr_matrix(b) for b in blocks], format="csr")

    def fit(self, df, y):
        X = self._matrix(df, fit=True)
        best = (np.inf, None)
        for a in self.alphas:
            pred = np.zeros(len(y))
            for tr, va in KFold(3, shuffle=True, random_state=self.seed).split(np.zeros(len(y))):
                pred[va] = Ridge(alpha=a, solver="lsqr").fit(X[tr], y[tr]).predict(X[va])
            if rmse(y, pred) < best[0]:
                best = (rmse(y, pred), a)
        self.alpha_ = best[1]
        self.model = Ridge(alpha=self.alpha_, solver="lsqr").fit(X, y)
        return self

    def predict(self, df):
        return self.model.predict(self._matrix(df))


class EmbNumSVR:
    """RBF-SVR on [sentence embedding | w * standardised numeric features]; C chosen by inner CV inside fit().
    n_emb=0 gives an SVR on the numeric (handcrafted) features alone."""

    def __init__(self, n_emb, num_weight=0.25, c_grid=(0.5, 1.0, 2.0, 5.0), epsilon=0.1, seed=SEED):
        self.n_emb, self.num_weight, self.c_grid, self.epsilon, self.seed = n_emb, num_weight, c_grid, epsilon, seed

    def _prep(self, X, fit=False):
        emb, num = X[:, :self.n_emb], X[:, self.n_emb:]
        if fit:
            self.scaler = StandardScaler().fit(num)
        return np.hstack([emb, self.num_weight * self.scaler.transform(num)])

    def fit(self, X, y):
        Z = self._prep(np.asarray(X, float), fit=True)
        best = (np.inf, None)
        for c in self.c_grid:
            pred = np.zeros(len(y))
            for tr, va in KFold(3, shuffle=True, random_state=self.seed).split(Z):
                pred[va] = SVR(C=c, epsilon=self.epsilon, gamma="scale").fit(Z[tr], y[tr]).predict(Z[va])
            if rmse(y, pred) < best[0]:
                best = (rmse(y, pred), c)
        self.C_ = best[1]
        self.svr_ = SVR(C=self.C_, epsilon=self.epsilon, gamma="scale").fit(Z, y)
        return self

    def predict(self, X):
        return self.svr_.predict(self._prep(np.asarray(X, float)))


# ---------------------------------------------------------------- registry of base models (with on-disk cache)
BASE = {}          # name -> {"oof", "test", "oof_rep", "modality", "family", "backbone"}
REFIT = {}         # name -> zero-arg callable returning IN-SAMPLE (refit on all rows) train predictions (Section 9b)
_REFIT_MEMO = {}


def _cache_file(name, tag, legacy=False):
    if legacy:                                          # v3 naming (kept so a finished DeBERTa run is reused)
        return CACHE_DIR / "predictions" / f"{name}__{CV_ID}__{tag}.npz"
    return CACHE_DIR / "predictions" / f"{name}__{CV_ID}__R{N_REPEATS}__{tag or 'notag'}__{PROBE_VERSION}.npz"


def _store(name, modality, family, backbone, oof, test, oof_rep, src):
    assert len(oof) == len(y) and len(test) == N_TEST and np.isfinite(oof).all() and np.isfinite(test).all(), name
    BASE[name] = {"oof": oof, "test": test, "oof_rep": oof_rep, "modality": modality, "family": family, "backbone": backbone}
    m = evaluate(y, oof)
    single = ""
    if oof_rep is not None and len(oof_rep) > 1:
        single = f" | single-repeat {np.mean([rmse(y, o) for o in oof_rep]):.4f}"
    print(f"{name:<30} [{modality:<5}] RMSE {m['rmse']:.4f}{single} | Pearson {m['pearson']:.4f} | QWK {m['qwk']:.4f}   ({src})")


def register(name, modality, compute, refit=None, tag="", family=None, legacy_cache=False):
    """Run (or load from cache) one base model; store OOF (repeat-averaged) / test predictions and print metrics."""
    if refit is not None:
        REFIT[name] = refit
    cache = _cache_file(name, tag, legacy=legacy_cache)
    if legacy_cache and not cache.exists():
        cache = _cache_file(name, tag)
    if cache.exists() and not FORCE_RECOMPUTE:
        d = np.load(cache)
        oof, test, rep, src = d["oof"], d["test"], (d["rep"] if "rep" in d.files else None), "cache"
    else:
        t0 = time.time()
        out = compute()
        oof, test, rep = out if len(out) == 3 else (*out, None)
        np.savez(cache, oof=oof, test=test, **({} if rep is None else {"rep": rep}))
        src = f"{time.time() - t0:.0f}s"
    _store(name, modality, family or name, name, oof, test, rep, src)


def register_bundle(key, modality, compute, X_all, tag="", prefix="probe"):
    """One cache file / one CV pass for a ProbeBundle -> one registered base model per head."""
    cache = _cache_file(f"{prefix}_{key}_bundle_{'-'.join(PROBE_HEADS)}", tag)
    if cache.exists() and not FORCE_RECOMPUTE:
        d = np.load(cache)
        res = {h: (d[f"{h}__oof"], d[f"{h}__test"], d[f"{h}__rep"]) for h in PROBE_HEADS if f"{h}__oof" in d.files}
        src = "cache"
    else:
        t0 = time.time()
        res = compute()
        np.savez(cache, **{f"{h}__{k}": v for h, trio in res.items() for k, v in zip(("oof", "test", "rep"), trio)})
        src = f"{time.time() - t0:.0f}s (all heads)"

    def memo_refit(h):
        def f():
            if key not in _REFIT_MEMO:
                _REFIT_MEMO[key] = ProbeBundle().fit(X_all, y).predict(X_all)
            return np.asarray(_REFIT_MEMO[key][h], float)
        return f

    for h, (oof, test, rep) in res.items():
        name = f"{prefix}_{key}_{h}"
        REFIT[name] = memo_refit(h)
        _store(name, modality, f"{prefix}_{key}", key, oof, test, rep, src)


def refit_fn(make, X):
    """In-sample predictions of a model refit on ALL rubric-valid rows (used only for the training-RMSE report)."""
    return lambda: np.asarray(make().fit(X, y).predict(X), float)

## 8. Base models

Each line below trains one model family with the repeated CV loop and prints its **OOF** RMSE (averaged over repeats; the `single-repeat` figure is what one random split would have reported), Pearson and quadratic-weighted kappa.

| Model | Input | Modality |
|---|---|---|
| `tfidf_ridge` | word + char TF-IDF, v1 numeric features | text |
| `mpnet_svr`, `bge_large_svr` | sentence embedding, v1 numeric features | text |
| `handcrafted_et`, `handcrafted_svr` | all interpretable features (fluency, LM surprise, syntax, ELECTRA…) — trees and kernel | text |
| `probe_<audio backbone>_{svr,ridge,krr,lstack}` | layers of WavLM / Whisper / HuBERT / wav2vec2 | **audio** |
| `probe_<text backbone>_{svr,ridge,krr,lstack}` | layers of RoBERTa / Qwen / DeBERTa-v3-large on the transcript | text |
| `deberta_ft` | fine-tuned DeBERTa-v3 regressor (one repeat) | text |

In [ ]:
tfidf_X = pd.concat([train_asr[["transcript"]].iloc[idx_nz].reset_index(drop=True),
                     F_train[BASE_NUM_COLS].iloc[idx_nz].reset_index(drop=True)], axis=1)
tfidf_Xt = pd.concat([test_asr[["transcript"]].reset_index(drop=True), F_test[BASE_NUM_COLS]], axis=1)
mp_X = np.hstack([mpnet_train[idx_nz], F_train[BASE_NUM_COLS].values[idx_nz]])
mp_Xt = np.hstack([mpnet_test, F_test[BASE_NUM_COLS].values])
et_X, et_Xt = F_train[ALL_NUM_COLS].iloc[idx_nz].reset_index(drop=True), F_test[ALL_NUM_COLS]
texts_nz = train_asr["transcript"].values[idx_nz]


def make_et():
    return make_pipeline(SimpleImputer(strategy="median"),
                         ExtraTreesRegressor(n_estimators=500, min_samples_leaf=3, max_features=0.4,
                                             n_jobs=-1, random_state=SEED))


print(f"Base models (OOF RMSE averaged over {N_REPEATS} repeat(s); 'single-repeat' = what one random split would give):")
register("tfidf_ridge", "text", lambda: run_cv(lambda: TfidfRidge(BASE_NUM_COLS), tfidf_X, y, tfidf_Xt),
         refit=refit_fn(lambda: TfidfRidge(BASE_NUM_COLS), tfidf_X), tag=TAG)
register("mpnet_svr", "text", lambda: run_cv(lambda: EmbNumSVR(mpnet_train.shape[1]), mp_X, y, mp_Xt),
         refit=refit_fn(lambda: EmbNumSVR(mpnet_train.shape[1]), mp_X), tag=TAG)
for ekey, (e_tr, e_te) in EXTRA_SENT.items():                       # v4: more sentence encoders (diversity)
    ex_X, ex_Xt = np.hstack([e_tr[idx_nz], F_train[BASE_NUM_COLS].values[idx_nz]]), np.hstack([e_te, F_test[BASE_NUM_COLS].values])
    register(f"{ekey}_svr", "text",
             lambda ex_X=ex_X, ex_Xt=ex_Xt, e_tr=e_tr: run_cv(lambda: EmbNumSVR(e_tr.shape[1]), ex_X, y, ex_Xt),
             refit=refit_fn(lambda e_tr=e_tr: EmbNumSVR(e_tr.shape[1]), ex_X), tag=TAG)
register("handcrafted_et", "text", lambda: run_cv(make_et, et_X, y, et_Xt),
         refit=refit_fn(make_et, et_X), tag=TAG)
# v4: a smooth (kernel) model on the same interpretable features - different inductive bias from the trees
register("handcrafted_svr", "text", lambda: run_cv(lambda: EmbNumSVR(0, num_weight=1.0), et_X.values, y, et_Xt.values),
         refit=refit_fn(lambda: EmbNumSVR(0, num_weight=1.0), et_X.values), tag=TAG)

for key, (A_tr, A_te) in AUDIO.items():
    register_bundle(key, "audio", lambda A_tr=A_tr, A_te=A_te: run_cv_bundle(ProbeBundle, A_tr[idx_nz], y, A_te),
                    X_all=A_tr[idx_nz])
for key, (T_tr, T_te) in TEXT_LAYERS.items():
    register_bundle(key, "text", lambda T_tr=T_tr, T_te=T_te: run_cv_bundle(ProbeBundle, T_tr[idx_nz], y, T_te),
                    X_all=T_tr[idx_nz], tag=TAG)

### 8a. Which layers carry grammar information?

Leave-one-out ridge error per layer, computed on all rubric-valid clips (descriptive — the models above re-select layers inside each fold). If the curve dips well below the *last-layer* value, that is exactly the information v1 threw away.

In [ ]:
curve_rows, curves = [], {}
for key, (A_tr, _) in {**AUDIO, **TEXT_LAYERS}.items():
    tab = score_layers(A_tr[idx_nz], y)
    curves[key] = tab.loc[tab.groupby("layer")["loo_rmse"].idxmin()].sort_values("layer").reset_index(drop=True)
    last = curves[key].iloc[-1]
    top = curves[key].loc[curves[key]["loo_rmse"].idxmin()]
    curve_rows.append({"backbone": key, "layers": len(curves[key]), "best_layer": int(top.layer), "best_pool": top.pool,
                       "best_LOO_rmse": top.loo_rmse, "last_layer_LOO_rmse": last.loo_rmse,
                       "gain_vs_last": last.loo_rmse - top.loo_rmse})
print(pd.DataFrame(curve_rows).round(4).to_string(index=False))

fig, axes = plt.subplots(1, len(curves), figsize=(min(4.4 * len(curves), 24), 3.4), sharey=True)
for ax, (key, cv) in zip(np.atleast_1d(axes), curves.items()):
    ax.plot(cv["layer"], cv["loo_rmse"], marker="o", ms=3, color="#4472C4")
    ax.axhline(y.std(), ls=":", color="gray", label="predict-the-mean")
    ax.scatter([cv["layer"].iloc[-1]], [cv["loo_rmse"].iloc[-1]], color="#C00000", zorder=5, label="last layer (v1 choice)")
    ax.set(title=key, xlabel="Layer")
    ax.legend(fontsize=7)
np.atleast_1d(axes)[0].set_ylabel("LOO ridge RMSE")
plt.tight_layout(); plt.show()

### 8b. Fine-tuned DeBERTa (single pass per fold)

v1 trained DeBERTa twice — once for OOF scores and again, with different batch size and learning rate, for test predictions. Here each fold model produces **both** its validation and its test predictions after every epoch; the epoch with the best OOF RMSE is used for both. Training is full FP32 (DeBERTa-v3 is unstable in FP16, as v1 found).
This is the slowest model and is **off by default in v5** (`RUN_DEBERTA = False`): the reference notebook measured 0.69 RMSE for it under speaker-grouped CV and dropped it. Set the flag to `True` to run it; the stacker simply does not have it otherwise.

In [ ]:
def run_deberta_cv():
    import torch.nn.functional as Fnn
    from torch.utils.data import DataLoader, Dataset
    from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup)
    cfg = DEBERTA
    tok = AutoTokenizer.from_pretrained(cfg["name"])
    enc_tr = tok(list(texts_nz), truncation=True, max_length=cfg["max_len"])
    enc_te = tok(test_asr["transcript"].tolist(), truncation=True, max_length=cfg["max_len"])
    collate = DataCollatorWithPadding(tok, pad_to_multiple_of=8)

    class DS(Dataset):
        def __init__(self, enc, idx, labels=None):
            self.enc, self.idx, self.labels = enc, idx, labels
        def __len__(self):
            return len(self.idx)
        def __getitem__(self, i):
            j = self.idx[i]
            item = {"input_ids": self.enc["input_ids"][j], "attention_mask": self.enc["attention_mask"][j]}
            if self.labels is not None:
                item["labels"] = float(self.labels[i])
            return item

    @torch.no_grad()
    def predict(model, loader):
        model.eval()
        out = []
        for batch in loader:
            batch.pop("labels", None)
            out.append(model(**{k: v.to(device) for k, v in batch.items()}).logits.view(-1).float().cpu().numpy())
        return np.concatenate(out)

    E = cfg["epochs"]
    oof_ep, test_ep = np.zeros((E, len(y))), np.zeros((E, N_TEST))
    te_loader = DataLoader(DS(enc_te, np.arange(N_TEST)), batch_size=cfg["batch"] * 2, collate_fn=collate)
    for k, (tr, va) in enumerate(folds):
        seed_everything(SEED + k)
        mu, sd = y[tr].mean(), y[tr].std()
        tr_loader = DataLoader(DS(enc_tr, tr, (y[tr] - mu) / sd), batch_size=cfg["batch"], shuffle=True, collate_fn=collate)
        va_loader = DataLoader(DS(enc_tr, va), batch_size=cfg["batch"] * 2, collate_fn=collate)
        model = AutoModelForSequenceClassification.from_pretrained(cfg["name"], num_labels=1).to(device).float()
        model.gradient_checkpointing_enable()
        no_decay = ("bias", "LayerNorm.weight")
        params = [{"params": [p for n, p in model.named_parameters() if not any(x in n for x in no_decay)], "weight_decay": 0.01},
                  {"params": [p for n, p in model.named_parameters() if any(x in n for x in no_decay)], "weight_decay": 0.0}]
        opt = torch.optim.AdamW(params, lr=cfg["lr"])
        total = math.ceil(len(tr_loader) / cfg["accum"]) * E
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
        for ep in range(E):
            model.train()
            opt.zero_grad(set_to_none=True)
            run_loss = 0.0
            for step, batch in enumerate(tr_loader):
                labels = batch.pop("labels").to(device).float()
                pred = model(**{kk: v.to(device) for kk, v in batch.items()}).logits.view(-1).float()
                loss = Fnn.mse_loss(pred, labels)
                (loss / cfg["accum"]).backward()
                run_loss += loss.item()
                if (step + 1) % cfg["accum"] == 0 or step + 1 == len(tr_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            oof_ep[ep, va] = predict(model, va_loader) * sd + mu
            test_ep[ep] += (predict(model, te_loader) * sd + mu) / len(folds)
            print(f"fold {k} epoch {ep + 1}: train-loss {run_loss / len(tr_loader):.3f} | val RMSE {rmse(y[va], oof_ep[ep, va]):.4f}")
        del model, opt, sched
        free_gpu()
    per_epoch = [rmse(y, oof_ep[e]) for e in range(E)]
    best = int(np.argmin(per_epoch))
    print("OOF RMSE per epoch:", np.round(per_epoch, 4), "-> using epoch", best + 1)
    return oof_ep[best], test_ep[best]


if RUN_DEBERTA:
    register("deberta_ft", "text", run_deberta_cv, tag=TAG, legacy_cache=True)   # reuses a finished v3 run if present

## 9. Stacking

The base models (now ~40 inputs) are highly correlated, so the stacker's job is mostly *variance control*. Sixteen strategies are compared, **all scored by nested CV that is repeated over every fold set**
(fit on the OOF predictions of four folds, predict the fifth):

| strategy | idea |
|---|---|
| `mean_affine`, `best_single_affine` | baselines: average of everything / best single model, each linearly calibrated |
| `nnls` | non-negative linear stack (v2/v3's choice) |
| `bagged_nnls` | NNLS averaged over 60 bootstrap resamples → stable weights |
| `family_nnls`, `family_bagged` | heads of one backbone are averaged first → **one weight per backbone** instead of one per head |
| `caruana` | greedy forward selection with replacement + affine calibration |
| `…+poly2`, `…+iso_half` | a monotone recalibration (quadratic / half-isotonic) fitted on *inner* out-of-fold stack predictions, to stretch the compressed tails |
| `batch_nnls` | **v5, reference-style:** one NNLS for the ≈45 s recording batch and one for all other clips (each fitted only on its own clips) |
| `batch_bagged_nnls`, `batch_family_nnls`, `batch_family_bagged` | **v5:** the same split, but each batch's prediction is blended 50/50 with a global stack — *partial pooling*, because the 45 s batch has only ≈100 graded clips |
| `batch_family_bagged_full` | split stacks without pooling, family-level and bagged |

The strategy with the lowest **test-mix-weighted** nested RMSE is used (`SELECT_BY="testmix"`: each recording batch is weighted by its share of the *test* set, because the test mix differs from the training mix; `"plain"` uses the ordinary nested RMSE). Predictions are clipped to `[CLIP_LO, 5]` (2.0 when grades below 2 are <1.5% of graded clips). Headline metrics are the **mean of the per-repeat metrics** (un-smoothed). Picking the best of 16 strategies is a mild selection effect (≈0.003–0.006 RMSE);
a bootstrap interval over clips is printed so differences of that size are not over-interpreted.

*Caveat shared with v1:* base-model OOF predictions of the other folds came from models that saw the held-out fold — standard for stacking, slightly optimistic.

In [ ]:
names = list(BASE)
P = np.column_stack([BASE[n]["oof"] for n in names])
P_test = np.column_stack([BASE[n]["test"] for n in names])
modality = np.array([BASE[n]["modality"] for n in names])
FAMILY = np.array([BASE[n]["family"] for n in names])      # heads of one backbone share a family
names_arr = np.array(names)


class NNLSStack:
    """Non-negative linear stack (+ intercept).

    fam   : optional array of family labels per input column; columns of one family are averaged first, so the stacker
            learns one weight per backbone instead of one per (backbone, head) -> far fewer free parameters.
    n_bags: >1 -> bootstrap-aggregated NNLS (average of the coefficients of n_bags resampled fits): same expected
            prediction, much more stable weights when many inputs are highly correlated."""

    def __init__(self, fam=None, n_bags=1, seed=SEED):
        self.fam, self.n_bags, self.seed = (None if fam is None else np.asarray(fam)), n_bags, seed
        self.labels_ = None if fam is None else list(dict.fromkeys(self.fam))

    def _reduce(self, P):
        if self.fam is None:
            return P
        return np.column_stack([P[:, self.fam == u].mean(1) for u in self.labels_])

    def fit(self, P, y):
        Z = self._reduce(P)
        if self.n_bags <= 1:
            m = LinearRegression(positive=True).fit(Z, y)
            self.coef_, self.intercept_ = m.coef_, m.intercept_
            return self
        rng, n = np.random.RandomState(self.seed), len(y)
        coefs, inters = [], []
        for _ in range(self.n_bags):
            idx = rng.randint(0, n, n)
            m = LinearRegression(positive=True).fit(Z[idx], y[idx])
            coefs.append(m.coef_); inters.append(m.intercept_)
        self.coef_, self.intercept_ = np.mean(coefs, 0), float(np.mean(inters))
        return self

    def predict(self, P):
        return self._reduce(P) @ self.coef_ + self.intercept_


class MeanAffine:
    """Plain average of all models, then a linear calibration."""
    def fit(self, P, y):
        self.m = LinearRegression().fit(P.mean(1, keepdims=True), y)
        return self
    def predict(self, P):
        return self.m.predict(P.mean(1, keepdims=True))


class BestSingleAffine:
    """Best single model (by training RMSE), linearly calibrated."""
    def fit(self, P, y):
        self.j = int(np.argmin([rmse(y, P[:, j]) for j in range(P.shape[1])]))
        self.m = LinearRegression().fit(P[:, [self.j]], y)
        return self
    def predict(self, P):
        return self.m.predict(P[:, [self.j]])


class Caruana:
    """Greedy forward ensemble selection with replacement (Caruana et al., 2004) + affine calibration.
    Very robust against over-fitting when there are many correlated candidate models."""

    def __init__(self, n_iter=60):
        self.n_iter = n_iter

    def fit(self, P, y):
        counts, total = np.zeros(P.shape[1]), np.zeros(len(y))
        for t in range(self.n_iter):
            err = (((total[:, None] + P) / (t + 1) - y[:, None]) ** 2).mean(0)
            j = int(np.argmin(err))
            counts[j] += 1
            total += P[:, j]
        self.w_ = counts / counts.sum()
        self.cal_ = LinearRegression().fit((P @ self.w_)[:, None], y)
        return self

    def predict(self, P):
        return self.cal_.predict((P @ self.w_)[:, None])


class Recal:
    """Base stacker followed by a monotone recalibration learned on INNER out-of-fold predictions of that base stacker
    (never on in-sample blends).  kind='poly2' (3 parameters) or 'iso' (isotonic; mix<1 blends it with the identity)."""

    def __init__(self, make_base, kind="poly2", mix=1.0, inner_splits=5):
        self.make_base, self.kind, self.mix, self.inner_splits = make_base, kind, mix, inner_splits

    def fit(self, P, y):
        self.base_ = self.make_base().fit(P, y)
        inner = np.zeros(len(y))
        for tr, va in KFold(self.inner_splits, shuffle=True, random_state=SEED).split(P):
            inner[va] = self.make_base().fit(P[tr], y[tr]).predict(P[va])
        if self.kind == "poly2":
            c = np.polyfit(inner, y, 2)
            slope = lambda z: 2 * c[0] * z + c[1]
            if slope(1.0) <= 0 or slope(5.0) <= 0 or (c[0] < 0 and abs(c[0]) > 0.15):     # must stay monotone on [1, 5]
                c = np.polyfit(inner, y, 1)
            self.c_ = c
        else:
            self.iso_ = IsotonicRegression(y_min=1.0, y_max=5.0, out_of_bounds="clip").fit(inner, y)
        return self

    def predict(self, P):
        z = self.base_.predict(P)
        if self.kind == "poly2":
            return np.polyval(self.c_, z)
        return self.mix * self.iso_.predict(z) + (1.0 - self.mix) * z


class BatchStack:
    """Separate stackers for the ~45 s recording batch and for all other clips, partially pooled with a global stacker.

    mix=1.0 : each clip is predicted only by the stacker fitted on its own batch (reference notebook).
    mix=0.5 : 50/50 blend of the batch-specific and the global stacker - fewer effective free parameters when a batch is small.
    Batches with fewer than `min_rows` training clips fall back to the global stacker.  `needs_batch` tells the helpers below
    to pass the batch codes (0 = the 45 s batch) to fit()/predict()."""
    needs_batch = True

    def __init__(self, make_base, mix=0.5, min_rows=40):
        self.make_base, self.mix, self.min_rows = make_base, mix, min_rows

    def fit(self, P, y_, batch):
        self.glob_ = self.make_base().fit(P, y_)
        self.sub_ = {}
        for flag in (True, False):
            rows = (batch == 0) == flag
            if rows.sum() >= self.min_rows:
                self.sub_[flag] = self.make_base().fit(P[rows], y_[rows])
        return self

    def predict(self, P, batch):
        out = np.asarray(self.glob_.predict(P), float).copy()
        for flag, m in self.sub_.items():
            rows = (batch == 0) == flag
            if rows.any():
                out[rows] = self.mix * np.asarray(m.predict(P[rows]), float) + (1.0 - self.mix) * out[rows]
        return out


def _fit_stack(s, P, y_, batch):
    return s.fit(P, y_, batch) if getattr(s, "needs_batch", False) else s.fit(P, y_)


def _predict_stack(s, P, batch):
    return s.predict(P, batch) if getattr(s, "needs_batch", False) else s.predict(P)


def stacker_factories(cols):
    """Strategy name -> zero-arg constructor, for the model columns `cols` (family labels follow the columns)."""
    fam = FAMILY[cols]
    out = {
        "mean_affine": MeanAffine,
        "best_single_affine": BestSingleAffine,
        "nnls": lambda: NNLSStack(),
        "bagged_nnls": lambda: NNLSStack(n_bags=60),
        "family_nnls": lambda: NNLSStack(fam=fam),
        "family_bagged": lambda: NNLSStack(fam=fam, n_bags=60),
        "caruana": Caruana,
        "bagged_nnls+poly2": lambda: Recal(lambda: NNLSStack(n_bags=30), "poly2"),
        "bagged_nnls+iso_half": lambda: Recal(lambda: NNLSStack(n_bags=30), "iso", 0.5),
        "family_bagged+poly2": lambda: Recal(lambda: NNLSStack(fam=fam, n_bags=30), "poly2"),
        "family_bagged+iso_half": lambda: Recal(lambda: NNLSStack(fam=fam, n_bags=30), "iso", 0.5),
    }
    if USE_BATCH_STACK:                                                    # v5: recording-batch-aware stackers
        out.update({
            "batch_nnls": lambda: BatchStack(lambda: NNLSStack(), mix=1.0),
            "batch_bagged_nnls": lambda: BatchStack(lambda: NNLSStack(n_bags=30), mix=0.5),
            "batch_family_nnls": lambda: BatchStack(lambda: NNLSStack(fam=fam), mix=0.5),
            "batch_family_bagged": lambda: BatchStack(lambda: NNLSStack(fam=fam, n_bags=30), mix=0.5),
            "batch_family_bagged_full": lambda: BatchStack(lambda: NNLSStack(fam=fam, n_bags=30), mix=1.0),
        })
    return out


def nested_oof(make_stacker, cols):
    """Stacker evaluated by CV on the base-model OOF matrix; repeated over every fold set. Returns one OOF vector per repeat."""
    outs = []
    for fs in FOLD_SETS:
        o = np.zeros(len(y))
        for tr, va in fs:
            s = _fit_stack(make_stacker(), P[tr][:, cols], y[tr], B_TR[tr])
            o[va] = np.clip(_predict_stack(s, P[va][:, cols], B_TR[va]), CLIP_LO, 5.0)
        outs.append(o)
    return outs


all_cols = np.arange(len(names))
nested = {k: nested_oof(mk, all_cols) for k, mk in stacker_factories(all_cols).items()}
rows = [{"strategy": k, **mean_metrics(y, v), "rmse_testmix": testmix_rmse(y, v)} for k, v in nested.items()]
fac_sel = lambda sel: stacker_factories(np.where(sel)[0])["family_nnls"]
for label, sel in (("audio models only", modality == "audio"), ("text models only", modality == "text")):
    if sel.any():
        _o = nested_oof(fac_sel(sel), np.where(sel)[0])
        rows.append({"strategy": f"family_nnls | {label}", **mean_metrics(y, _o), "rmse_testmix": testmix_rmse(y, _o)})
rows += [{"strategy": f"{n} (single model, raw)", **evaluate(y, BASE[n]["oof"]), "rmse_testmix": testmix_rmse(y, [BASE[n]["oof"]])} for n in names]
SORT_COL = "rmse_testmix" if SELECT_BY == "testmix" else "rmse"
stack_table = pd.DataFrame(rows).sort_values(SORT_COL).reset_index(drop=True)
_ref_rows = pd.DataFrame([
    {"strategy": "v1 final (random folds, reported)", "rmse": V1_REFERENCE["rmse"], "pearson": V1_REFERENCE["pearson"]},
    {"strategy": "v2 final (random folds, reported)", "rmse": V2_REFERENCE["rmse"], "pearson": V2_REFERENCE["pearson"]},
    {"strategy": "reference notebook (speaker-grouped CV, reported)", "rmse": REFERENCE_NB["grouped_cv_rmse"], "pearson": REFERENCE_NB["pearson"]}])
print(f"Nested CV ({CV_MODE} folds), sorted by {SORT_COL}; 'rmse_testmix' re-weights recording batches to the TEST mix:")
print(stack_table.round(4).head(30).to_string(index=False))
print(f"... ({len(stack_table)} rows in total; the full table is saved with the results)")
print("\nReported numbers for reference (v1/v2 used RANDOM folds and are not comparable with speaker-grouped CV):")
print(_ref_rows.round(4).to_string(index=False))
stack_table = pd.concat([stack_table, _ref_rows], ignore_index=True)

STRATEGY_FACTORIES = stacker_factories(all_cols)
_select_score = (lambda k: testmix_rmse(y, nested[k])) if SELECT_BY == "testmix" else (lambda k: np.mean([rmse(y, o) for o in nested[k]]))
FINAL_STRATEGY = min(nested, key=_select_score)
final_stack = _fit_stack(STRATEGY_FACTORIES[FINAL_STRATEGY](), P, y, B_TR)
final_oof_reps = nested[FINAL_STRATEGY]
final_oof = np.mean(final_oof_reps, axis=0)                       # for plots only
final_metrics = mean_metrics(y, final_oof_reps)                    # headline: mean of per-repeat metrics (un-smoothed)
final_test = np.clip(_predict_stack(final_stack, P_test, B_TE), CLIP_LO, 5.0)      # model output; the grade-0 gate is applied in Section 10

print(f"\nSelected strategy: {FINAL_STRATEGY}   ({len(names)} base models, {len(set(FAMILY))} families)")
FINAL_TESTMIX = testmix_rmse(y, final_oof_reps)
print("Nested OOF (mean over repeats):", {k: round(v, 4) for k, v in final_metrics.items()}, f"| test-mix-weighted RMSE {FINAL_TESTMIX:.4f}")
print(f"vs reference notebook (speaker-grouped CV, graded clips): RMSE {REFERENCE_NB['grouped_cv_rmse']:.4f} / Pearson {REFERENCE_NB['pearson']:.4f} "
      f"-> yours {final_metrics['rmse']:.4f} / {final_metrics['pearson']:.4f}")
if CV_MODE == "random":
    print(f"vs v2 (random folds): RMSE {V2_REFERENCE['rmse']:.4f} / Pearson {V2_REFERENCE['pearson']:.4f} -> {final_metrics['rmse']:.4f} / {final_metrics['pearson']:.4f}")
else:
    print(f"NOTE: CV_MODE='{CV_MODE}' is stricter than v1/v2's random folds - do NOT compare with v2's {V2_REFERENCE['rmse']:.4f}.")
print("Note: choosing the best of", len(nested), f"strategies by nested {SORT_COL} is a mild selection effect (≈0.003-0.006).")

# bootstrap uncertainty of the headline number (resampling clips)
_rng = np.random.RandomState(SEED)
_boot = [rmse(y[i], final_oof[i]) for i in (_rng.randint(0, len(y), len(y)) for _ in range(2000))]
RMSE_CI = (float(np.percentile(_boot, 2.5)), float(np.percentile(_boot, 97.5)))
print(f"RMSE 95% bootstrap interval (clips resampled): [{RMSE_CI[0]:.4f}, {RMSE_CI[1]:.4f}]  <- compare v2's {V2_REFERENCE['rmse']:.4f} with this width")

## 9b. Training-data RMSE (required by the competition) next to the honest estimate

The competition asks for the RMSE **on the training data**. Two different numbers answer that, and they must not be confused:

* **Training (resubstitution) RMSE** — every base model is re-fitted on *all* rubric-valid training clips, predicts those same clips, and the final stack is applied. This is optimistic by construction (the models have seen these labels) and is reported only because it is required.
* **Out-of-fold (nested-CV) RMSE** — each clip is predicted by models that never saw it. **This is the number to use for judging quality / expected leaderboard performance.**

Clips with label `0` are outside the 1–5 rubric and are excluded from model training. The first pair of numbers is computed on the ≈732 rubric-valid clips; v5 additionally reports the figure **over all 769 training clips**, in which the grade-0 gate (Section 3b) sets the loud zero-grade clips to 0. Models without a refit callable (e.g. DeBERTa) fall back to their out-of-fold prediction in the training column, which makes the training RMSE slightly *less* optimistic than a pure refit.

In [ ]:
resub_cols, fallbacks = [], []
for n in names:
    if n in REFIT:
        resub_cols.append(REFIT[n]())
    else:
        resub_cols.append(BASE[n]["oof"]); fallbacks.append(n)
P_resub = np.column_stack(resub_cols)
train_pred = np.clip(_predict_stack(final_stack, P_resub, B_TR), CLIP_LO, 5.0)

train_metrics = evaluate(y, train_pred)
oof_metrics = evaluate(y, final_oof)
summary = pd.DataFrame({"training (resubstitution)": train_metrics, "out-of-fold (nested CV)": oof_metrics}).T
print(f"Rubric-valid training clips: {len(y)}   (clips with label 0 excluded: {int((~nz_mask).sum())})")
if fallbacks:
    print("Base models using OOF predictions in the training column (no refit available):", fallbacks)
print(summary.round(4).to_string())
print(f"\n>>> TRAINING RMSE (final model) = {train_metrics['rmse']:.4f}   |   OUT-OF-FOLD RMSE = {oof_metrics['rmse']:.4f}")
print("Training RMSE is optimistic; the out-of-fold figure is the honest estimate of unseen-data error.")
TRAIN_RMSE = train_metrics["rmse"]

# v5: the same two figures over ALL training clips, with the grade-0 gate setting the loud zero-grade clips to 0
full_train, full_cv = np.zeros(N_TRAIN), np.zeros(N_TRAIN)
full_train[idx_nz], full_cv[idx_nz] = train_pred, final_oof
full_train[GATE[:N_TRAIN]] = 0.0
full_cv[GATE[:N_TRAIN]] = 0.0
TRAIN_RMSE_ALL, CV_RMSE_ALL = rmse(full_train, lab_all), rmse(full_cv, lab_all)
print(f">>> over all {N_TRAIN} training clips (gate applied): TRAINING RMSE = {TRAIN_RMSE_ALL:.4f} | OUT-OF-FOLD RMSE = {CV_RMSE_ALL:.4f} "
      f"| predicting the mean: {rmse(np.full(N_TRAIN, lab_all.mean()), lab_all):.4f}")

## 10. Final predictions and submission

Test predictions are the stacker applied to each base model's **fold-averaged** test prediction, clipped to `[CLIP_LO, 5]`; test clips above the grade-0 gate threshold (Section 3b) are then set to 0. The file follows the column names of `sample_submission.csv` and the validated `test.csv` order.

In [ ]:
id_col = sample_df.columns[0]
label_col = sample_df.columns[1] if sample_df.shape[1] > 1 else "label"
GATE_TEST = GATE[N_TRAIN:]
final_test_sub = final_test.copy()
final_test_sub[GATE_TEST] = 0.0                                   # grade-0 gate (Section 3b); no-op when the gate is off
submission = pd.DataFrame({id_col: test_df["filename"].values, label_col: final_test_sub})

assert len(submission) == len(test_df) == N_TEST
assert submission[id_col].tolist() == test_df["filename"].tolist()
assert submission[id_col].is_unique
assert submission[label_col].between(0.0, 5.0).all() and submission[label_col].notna().all()
assert (submission[label_col].values[~GATE_TEST] >= CLIP_LO - 1e-9).all()

submission.to_csv(OUT_DIR / "submission.csv", index=False)
if set(sample_df[id_col]) != set(test_df["filename"]):
    alt = sample_df[[id_col]].merge(submission, on=id_col, how="left")
    if alt[label_col].notna().all():
        alt.to_csv(OUT_DIR / "submission_sample_order.csv", index=False)
        print(f"sample_submission has {len(sample_df)} rows -> also wrote submission_sample_order.csv")
    else:
        print("sample_submission contains file names that are not in test.csv; only the test.csv-order file was written.")

pd.DataFrame({"filename": train_df["filename"].iloc[idx_nz].values, "label": y, "oof_pred": final_oof,
              "fold": fold_id}).to_csv(OUT_DIR / "oof_predictions.csv", index=False)
pd.DataFrame({n: BASE[n]["oof"] for n in names}).assign(label=y).to_csv(OUT_DIR / "base_models_oof.csv", index=False)
pd.DataFrame({n: BASE[n]["test"] for n in names}).assign(filename=test_df["filename"].values).to_csv(
    OUT_DIR / "base_models_test.csv", index=False)
stack_table.to_csv(OUT_DIR / "stack_table.csv", index=False)
json.dump({"version": "v5", "cv_mode": CV_MODE, "cv_id": CV_ID, "n_repeats": N_REPEATS, "strategy": FINAL_STRATEGY, "select_by": SELECT_BY,
           "nested_oof": final_metrics, "nested_rmse_testmix": FINAL_TESTMIX, "rmse_95ci": RMSE_CI,
           "training_rmse": TRAIN_RMSE, "training_metrics": train_metrics,
           "training_rmse_all_clips": TRAIN_RMSE_ALL, "cv_rmse_all_clips": CV_RMSE_ALL,
           "zero_gate": {"threshold": GATE_THR, "gated_test_clips": test_df["filename"][GATE_TEST].tolist(), "clip_lo": CLIP_LO},
           "v1_reference": V1_REFERENCE, "v2_reference": V2_REFERENCE, "reference_notebook": REFERENCE_NB,
           "models": names, "audio_backbones": list(AUDIO), "transcript_tag": TAG},
          open(OUT_DIR / "run_summary.json", "w"), indent=2)

print(f"gated (set to 0) test clips: {test_df['filename'][GATE_TEST].tolist()}")
print(submission[label_col].describe().round(3).to_string())
print("\nSaved to", OUT_DIR)
print(" -", "submission.csv (upload this)")

## 11. Results and interpretation

In [ ]:
oof_true_r = np.floor(y + 0.5).astype(int)
oof_pred_r = np.floor(np.clip(final_oof, 1, 5) + 0.5).astype(int)
cm = pd.crosstab(oof_true_r, oof_pred_r).reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)
cm_norm = cm.div(cm.sum(1).replace(0, 1), axis=0)
jit = np.random.RandomState(SEED).uniform(-0.08, 0.08, len(y))

fig, ax = plt.subplots(2, 2, figsize=(11.5, 8.5))
ax[0, 0].scatter(y + jit, final_oof, s=16, alpha=0.45, color="#4472C4")
ax[0, 0].plot([1, 5], [1, 5], "k--", lw=1)
ax[0, 0].set(title=f"OOF: observed vs predicted (RMSE {final_metrics['rmse']:.3f}, r {final_metrics['pearson']:.3f})",
             xlabel="Observed grammar score", ylabel="Predicted")
ax[0, 1].scatter(final_oof, y - final_oof, s=16, alpha=0.45, color="#ED7D31")
ax[0, 1].axhline(0, color="k", ls="--", lw=1)
ax[0, 1].set(title="OOF residuals", xlabel="Predicted", ylabel="Observed − predicted")
im = ax[1, 0].imshow(cm_norm.values, cmap="Blues", vmin=0, vmax=1)
ax[1, 0].set(title="Confusion (rounded scores, row-normalised)", xlabel="Predicted", ylabel="Observed",
             xticks=range(5), xticklabels=range(1, 6), yticks=range(5), yticklabels=range(1, 6))
for i in range(5):
    for j in range(5):
        ax[1, 0].text(j, i, int(cm.values[i, j]), ha="center", va="center",
                      color="white" if cm_norm.values[i, j] > 0.55 else "black", fontsize=9)
ax[1, 1].hist(final_oof, bins=20, alpha=0.65, density=True, label="OOF (train)", color="#4472C4")
ax[1, 1].hist(final_test, bins=20, alpha=0.65, density=True, label="test predictions", color="#70AD47")
ax[1, 1].set(title="Prediction distribution: train OOF vs test", xlabel="Predicted score"); ax[1, 1].legend()
plt.tight_layout(); plt.show()

per_label = pd.DataFrame({"label": y, "pred": final_oof, "abs_err": np.abs(y - final_oof)}).groupby("label").agg(
    n=("label", "size"), mean_pred=("pred", "mean"), MAE=("abs_err", "mean")).round(3)
print("Error by true score (regression toward the middle at the extremes is typical for small, imbalanced data):")
print(per_label.to_string())

In [ ]:
model_rmse = pd.Series({n: rmse(y, BASE[n]["oof"]) for n in names}).sort_values()
fold_rmse = [rmse(y[va], final_oof[va]) for _, va in folds]
color_of = {"audio": "#ED7D31", "text": "#4472C4"}
fam_modality = {BASE[n]["family"]: BASE[n]["modality"] for n in names}


def stack_weights(stacker):
    """(weights, intercept) of the fitted stacker's first stage - whichever strategy won. A DataFrame (one column per
    recording batch) for batch-split stackers, otherwise a Series."""
    s = stacker.base_ if isinstance(stacker, Recal) else stacker
    if isinstance(s, BatchStack):
        g_w, g_b = stack_weights(s.glob_)
        cols, inter = {}, {}
        for flag, label in ((True, "45 s batch"), (False, "other clips")):
            if flag in s.sub_:
                w, b = stack_weights(s.sub_[flag])
                cols[label], inter[label] = s.mix * w + (1.0 - s.mix) * g_w, s.mix * b + (1.0 - s.mix) * g_b
            else:
                cols[label], inter[label] = g_w, g_b
        return pd.DataFrame(cols), float(inter["45 s batch"])
    if isinstance(s, NNLSStack):
        return pd.Series(s.coef_, index=s.labels_ if s.labels_ is not None else names), float(s.intercept_)
    if isinstance(s, Caruana):
        return pd.Series(s.w_, index=names), float(s.cal_.intercept_)
    if isinstance(s, MeanAffine):
        return pd.Series(np.full(len(names), s.m.coef_[0] / len(names)), index=names), float(s.m.intercept_)
    return None, None


weights, w_int = stack_weights(final_stack)
if weights is not None:             # aggregate to backbone families so the plot stays readable with 40+ inputs
    fam_of = {n: BASE[n]["family"] for n in names}
    weights = weights.groupby(lambda k: fam_of.get(k, k)).sum()
    weights = weights.sort_values(weights.columns[0]) if isinstance(weights, pd.DataFrame) else weights.sort_values()

fig, ax = plt.subplots(1, 3, figsize=(17, 6.2))
top = model_rmse.head(22)[::-1]
ax[0].barh(top.index, top.values, color=[color_of[BASE[n]["modality"]] for n in top.index])
ax[0].axvline(final_metrics["rmse"], color="k", ls="--", label=f"stack {final_metrics['rmse']:.3f}")
ax[0].axvline(y.std(), color="gray", ls=":", label="predict-the-mean")
ax[0].set(title="22 best base models (orange = audio, blue = text)", xlabel="OOF RMSE"); ax[0].legend(fontsize=8)
ax[0].tick_params(axis="y", labelsize=7)
if weights is not None:
    if isinstance(weights, pd.DataFrame):          # batch-split stack: one bar per recording batch (reference: weights differ strongly)
        weights.plot.barh(ax=ax[1], width=0.8, color=["#C00000", "#2E75B6"][:weights.shape[1]])
        ax[1].set(title=f"{FINAL_STRATEGY}: weight per backbone family\n(45 s batch: intercept {w_int:+.2f}, sum {weights.iloc[:, 0].sum():.2f})", xlabel="Weight")
        ax[1].legend(fontsize=7)
    else:
        ax[1].barh(weights.index, weights.values, color=[color_of.get(fam_modality.get(k), "#7F7F7F") for k in weights.index])
        ax[1].set(title=f"{FINAL_STRATEGY}: weight per backbone family\n(intercept {w_int:+.2f}, sum {weights.sum():.2f})", xlabel="Weight")
    ax[1].tick_params(axis="y", labelsize=7)
else:
    ax[1].text(0.5, 0.5, f"{FINAL_STRATEGY}\n(no per-model weights)", ha="center", va="center"); ax[1].axis("off")
ax[2].bar(range(N_SPLITS), fold_rmse, color="#70AD47")
ax[2].axhline(final_metrics["rmse"], color="k", ls="--")
ax[2].set(title="RMSE by fold (repeat 0 split; stability)", xlabel="Fold", ylabel="RMSE")
plt.tight_layout(); plt.show()

In [ ]:
# ---- v5: error by recording batch, and the headline numbers on the scale that matters for the test set ----
rows = []
for b in range(4):
    m = B_TR == b
    if m.sum() > 2:
        rows.append({"batch": BATCH_NAMES[b], "train clips": int(m.sum()), "train share": m.mean(), "test share": TEST_SHARE[b],
                     "mean grade": y[m].mean(), "RMSE": float(np.mean([rmse(y[m], o[m]) for o in final_oof_reps])),
                     "Pearson": pearsonr(y[m], final_oof[m])[0]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

print(f"\nNested RMSE, graded clips (training mix)          : {final_metrics['rmse']:.4f}")
print(f"Nested RMSE, re-weighted to the TEST batch mix    : {FINAL_TESTMIX:.4f}")
print(f"Nested RMSE over all {N_TRAIN} clips (gate applied)      : {CV_RMSE_ALL:.4f}   <- same definition as the reference's 'CV, all clips'")
print(f"Reference notebook, speaker-grouped CV (graded)   : {REFERENCE_NB['grouped_cv_rmse']:.4f}  (Pearson {REFERENCE_NB['pearson']:.4f})")
if CV_MODE == "random":
    print("WARNING: CV_MODE='random' leaks speakers - these figures are optimistic and not comparable with the reference.")

In [ ]:
# ---- v4: which backbone families actually earn their place?  (leave-one-family-out, nested CV, family-level NNLS) ----
fam_list = list(dict.fromkeys(FAMILY))
ref_rmse = np.mean([rmse(y, o) for o in nested_oof(lambda: NNLSStack(fam=FAMILY), all_cols)])
abl = {}
for f in fam_list:
    cols = np.where(FAMILY != f)[0]
    abl[f] = np.mean([rmse(y, o) for o in nested_oof(lambda cols=cols: NNLSStack(fam=FAMILY[cols]), cols)]) - ref_rmse
abl = pd.Series(abl).sort_values()
print(f"Reference (all {len(fam_list)} families, family-NNLS): nested RMSE {ref_rmse:.4f}")
print("Change in nested RMSE when a family is REMOVED (positive = the family helps; ≈0 = redundant):")
print(abl.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, max(3.0, 0.28 * len(abl) + 1)))
ax.barh(abl.index, abl.values, color=[color_of.get(fam_modality.get(f), "#7F7F7F") for f in abl.index])
ax.axvline(0, color="k", lw=1)
ax.set(title="Leave-one-family-out: ΔRMSE of the stack (positive = family helps)", xlabel="Δ nested RMSE")
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ---- which interpretable features matter? ----
corr = pd.Series({c: spearmanr(F_train[c].values[idx_nz], y)[0] for c in ALL_NUM_COLS}).dropna()
top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(14)[::-1]
et_full = ExtraTreesRegressor(n_estimators=500, min_samples_leaf=3, max_features=0.4, n_jobs=-1,
                              random_state=SEED).fit(et_X, y)
imp = pd.Series(et_full.feature_importances_, index=ALL_NUM_COLS).sort_values(ascending=False).head(14)[::-1]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].barh(top_corr.index, top_corr.values, color=["#C00000" if v < 0 else "#2E75B6" for v in top_corr.values])
ax[0].set(title="Spearman correlation with grammar score", xlabel="ρ")
ax[1].barh(imp.index, imp.values, color="#7F7F7F")
ax[1].set(title="ExtraTrees feature importance (all data)", xlabel="Importance")
plt.tight_layout(); plt.show()

# ---- largest errors, with transcripts ----
worst = np.argsort(-np.abs(y - final_oof))[:6]
print("Largest OOF errors:")
for i in worst:
    row = train_asr.iloc[idx_nz[i]]
    print(f"\n{row.filename} | true {y[i]:g} | predicted {final_oof[i]:.2f} | words {int(row.word_count)}")
    print("  ", row.transcript[:300])

## 12. Robustness checks

**a) Speaker-leakage check.** Audio encoders can encode speaker identity. The best audio backbone (SVR head) is re-scored with one split of the *other* kind (random folds when the run uses speaker-grouped folds, speaker-grouped folds otherwise). The reference saw a Whisper-encoder SVR move from 0.522 (random folds) to 0.600 (speaker folds); a large gap confirms why Section 7 groups the folds.
**b) Zero-label regime — second opinion.** The grade-0 gate of Section 3b is a loudness rule. This section trains an independent detector (WavLM statistics + ASR confidence) and lists test clips it flags, so a disagreement with the gate is visible.

In [ ]:
audio_fams = sorted({BASE[n]["backbone"] for n in names if BASE[n]["modality"] == "audio" and BASE[n]["family"].startswith("probe_")})
if audio_fams:
    head = "svr" if "svr" in PROBE_HEADS else PROBE_HEADS[0]
    best_key = min(audio_fams, key=lambda k: rmse(y, BASE[f"probe_{k}_{head}"]["oof"]))
    alt_mode = "random" if CV_MODE != "random" else "speaker"
    a_res = run_cv_bundle(lambda: ProbeBundle((head,)), AUDIO[best_key][0][idx_nz], y, AUDIO[best_key][1],
                          fold_sets=[make_folds(alt_mode)])
    a_rmse = rmse(y, a_res[head][0])
    rep = BASE[f"probe_{best_key}_{head}"]["oof_rep"]
    cur_rmse = float(np.mean([rmse(y, o) for o in rep])) if rep is not None else rmse(y, BASE[f"probe_{best_key}_{head}"]["oof"])
    by_mode = {CV_MODE: cur_rmse, alt_mode: a_rmse}
    print(f"probe_{best_key}_{head}: {CV_MODE} folds RMSE {cur_rmse:.4f}  |  {alt_mode} folds (single split) RMSE {a_rmse:.4f}")
    if "speaker" in by_mode and "random" in by_mode:
        gap = by_mode["speaker"] - by_mode["random"]
        print(f"leakage gap (speaker - random) = {gap:+.4f}: " + (
            "small - little speaker leakage." if gap < 0.03 else
            "random folds are clearly optimistic for audio models; keep CV_MODE='speaker' (the reference saw ~+0.08)."))
else:
    print("Skipped (no audio probe models).")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict

bk = next(iter(AUDIO))
A_tr_z, A_te_z = AUDIO[bk]
Dz = A_tr_z.shape[2] // 2
quality = ["mean_word_probability", "avg_logprob", "language_probability"]
Xz_tr = np.hstack([A_tr_z[:, :, :Dz].mean(1), train_asr[quality].fillna(0).values])
Xz_te = np.hstack([A_te_z[:, :, :Dz].mean(1), test_asr[quality].fillna(0).values])
is_zero = (~nz_mask).astype(int)

if 0 < is_zero.sum() < len(is_zero):
    detector = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, class_weight="balanced", max_iter=2000))
    oof_p = cross_val_predict(detector, Xz_tr, is_zero, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                              method="predict_proba")[:, 1]
    detector.fit(Xz_tr, is_zero)
    p_test = detector.predict_proba(Xz_te)[:, 1]
    print(f"zero-regime detector ({bk} + ASR confidence): CV ROC-AUC {roc_auc_score(is_zero, oof_p):.3f}")
    print(f"test clips with P(zero-regime) > 0.5: {(p_test > 0.5).sum()} | > 0.9: {(p_test > 0.9).sum()} (of {N_TEST})")
    if (p_test > 0.5).any():
        flagged = pd.DataFrame({"filename": test_df["filename"], "p_zero": p_test, "model_pred": final_test, "loudness_gate": GATE[N_TRAIN:]})
        print(flagged.sort_values("p_zero", ascending=False).head(8).round(3).to_string(index=False))
        print("'loudness_gate' = True means the Section 3b gate sets that clip to 0; for the others the model predicts >= CLIP_LO. Inspect any disagreement manually.")
    else:
        print("The independent detector flags no test clip; the loudness gate decides alone.")
else:
    print("No zero-label rows in train.csv; nothing to check.")

## 13. Limitations and next steps

* **Small data (≈730 rubric-valid clips).** Fold-to-fold RMSE varies (fold chart); differences of ~0.01 between configurations are within noise — see the bootstrap interval printed in Section 9.
* **Selection effects are contained, not eliminated.** Layers, `k`, `C`, `γ`, `α` are chosen inside each training fold and the stacker is evaluated by repeated nested CV, but the *list* of backbones, heads and strategies was chosen by a person, and the strategy winner is picked on the same nested scores (≈0.003–0.005).
* **Not yet run on real data.** Everything in the "Status" box at the top applies. The new Whisper-decoder extractor in particular was only code-reviewed; if it fails it is skipped. If a new backbone does not help (see the leave-one-family-out chart) delete it from the config to save time; the stack will not miss it.
* **DeBERTa fine-tuning** is off by default (reference: 0.69 RMSE under grouped CV). When enabled it uses a single CV repeat, so its OOF is noisier than the repeat-averaged probes.
* **Whisper normalises speech.** Its language model tends to "repair" grammar, which hides exactly what is being scored. `VERBATIM_PROMPT = True` is an untested option (needs a full re-transcription and invalidates every transcript cache). The audio probes partly compensate because they never use the transcript.
* **Speaker groups are inferred**, not given: they come from voice similarity (Section 7). A speaker whose clips were not linked can still leak a little; the reference makes the same assumption.
* **Test mix.** ≈half of the test clips come from the 45 s batch, which is only ≈14% of the graded training clips. Model/stacker selection uses a test-mix-weighted RMSE, but it rests on ≈100 clips of that batch, so it is noisy; compare `rmse` and `rmse_testmix` before trusting a small difference.
* **Zero-label clips** are not modelled; the loudness gate (Section 3b) handles them. It rests on one observed regularity (zero-grade training clips are loud throughout) and on a handful of test clips — check the gated list. Set `USE_ZERO_GATE = False` to disable it.
* **Reference ideas not ported (need new GPU code):** Voxtral and Qwen2-Audio audio-token hidden states (the reference's best single models, CV RMSE ≈ 0.55), Qwen2.5-7B/14B hidden states with rubric prompts and score-digit probabilities, and a Parakeet second ASR. The reference reports ≈8 GPU-hours for its full extraction.
* **Ideas not implemented:** LoRA fine-tuning of a speech encoder, a CTC (verbatim) second ASR and its disagreement with Whisper as a feature, segment-level training of the probes, pseudo-labelling the test audio, a grammar-error-detection model trained on GEC corpora, a persisted `predict(audio_path)` function (no trained models are saved; everything re-runs from caches).

In [ ]:
# Kaggle: the file that is submitted must be /kaggle/working/submission.csv
import shutil
shutil.copy2(OUT_DIR / "submission.csv", KAGGLE_WORK_DIR / "submission.csv")
print("Final submission ->", KAGGLE_WORK_DIR / "submission.csv")
print(pd.read_csv(KAGGLE_WORK_DIR / "submission.csv").head())
